# JUICE RPWI HF SID2 (RAW): L1a Mask Formation for ASW1/2/3 -- 2026/6/8

In [ ]:
import copy
import csv
import math
import matplotlib.colors as colors
import matplotlib.pyplot as plt
import numpy as np
import os
import sys
from spacepy import pycdf

In [ ]:
sys.path.append('./lib/')
import juice_cal_lib   as juice_cal
import juice_cdf_lib   as juice_cdf
import juice_sid2_lib  as juice_sid2
import juice_spec_lib  as juice_spec

# Mode seting

In [ ]:
# *** Plot dump ***
dump_mode = 0                           # 0: no-dump  1:plot dump  2:comparison for operation
# *** CAL ***
unit_mode = 0                           # [Power]     0: raw     1: V＠ADC     2: V@HF    3: V@RWI  4: V/m
band_mode = 0                           # [Power]     0: sum     1: /Hz
cal_mode  = 2                           # [Power]     0: background     1: CAL     2: all
comp_mode = 0                           # [Complex]   0: Poweer  1: Matrix   3: Matrix-2D  >3: any
# *** Unit ***
f_mode    = 1                           # [Frequency] 0: linear  1: log
time_mode = 1                           # [Time]      0: Epoch   1: data number
gap_mode  = 0                           # [gap]       0: n/a     1: gap
# *** Frenquency in Linear ***
f_mode_min = 80;  f_mode_max = 2068     # 80 - 2068 : as same as SID-4/20
# *** Power range ***
p_raw_max = 7.0                         # background: 7.5   CAL: 10
p_raw_min = -1.0                        # background: 2.5   CAL: 5
# *** Directory set: set by User ***
work_dir = '/Users/user/0-python/JUICE_python/ql/'   # Plot dump folder

### MASK special

In [ ]:
# *** Filter make modes ***
asw_ver      = 3.1          # 1.0 / 2.0 / 3.0 / 3.1 / 3.2  for mask making
sid3_diff    = 2            # [dB]
filter_mode  = 0            # 0: check  1: make MASK            "lib/hf_sid3_mask-??k.csv"
filter_sid3  = 0            # 0: check  1: make freq-table      "lib/hf_sid3_freq-??k.csv"

# **** Mask file set ***
if asw_ver == 2.0:
    n_freq_msk = 19392;  f_min_msk = 80.0;   f_step_msk = 2.3125
else:
    n_freq_msk = 19360;  f_min_msk = 19.5;   f_step_msk = 2.3125

# get CDF data

In [ ]:
# date='20250331';  ver = 'V01'   # CAL
date='0';         ver = 'V02'   # Test data
data_dir, data_name_list = juice_sid2.datalist(date, ver)       # [date]   yyyymmdd: group read    others: file list

In [ ]:
class struct:
    pass

data = struct()
num_list = len(data_name_list)
for i in range(num_list):
    data_name = data_name_list[i];  cdf_file = data_dir + data_name;  print(i, cdf_file)
    cdf = pycdf.CDF(cdf_file);      data1 = juice_sid2.hf_sid2_read(cdf)
    if i==0:
        data = data1;                                print(data.Eu_i.shape)
    else:
        data = juice_sid2.hf_sid2_add(data, data1);  print(data.Eu_i.shape)
data_name = os.path.split(data_name)[1];             print("data name:", data_name)

In [ ]:
data = juice_sid2.hf_sid2_shaping(data, cal_mode)

In [ ]:
date1 = data.epoch[0];  date1 = date1.strftime('%Y/%m/%d %R:%S')
date2 = data.epoch[-1]; date2 = date2.strftime('%Y/%m/%d %R:%S')
str_date = date1 + "  -  " + date2
f_min0  = data.frequency[0][0][0]; f_max0  = max(np.ravel(data.frequency))

print("       date and time:", str_date)
print("         Num-samples:", data.n_time, "  Num-Frequency:", data.n_step, "  Length:", data.n_samp)
print("         Time-length:", data.time[0][0][-1], "sec in 1-sweep, with time resolution of ", data.time[0][0][1], "sec")
print("f, width, step (kHz):", f_min0,  "-", f_max0, data.freq_width[0][0][0], data.freq_step[0][0][0])

In [ ]:
n_sweep1 = 0;  n_sweep2 = data.n_time//2;  n_sweep3 = -1
print(f"[First peak - {n_sweep1}]")
peak_E = np.ravel(data.Eu_i[n_sweep1]); peak_f = np.nanargmax(peak_E); print("Eu:", '{:.2e} '.format(peak_E[peak_f]))
peak_E = np.ravel(data.Eu_i[n_sweep1]); peak_f = np.nanargmin(peak_E); print("Eu:", '{:.2e} '.format(peak_E[peak_f]))
print(f"[Mid   peak - {n_sweep2}]")
peak_E = np.ravel(data.Eu_i[n_sweep2]); peak_f = np.nanargmax(peak_E); print("Eu:", '{:.2e} '.format(peak_E[peak_f]))
peak_E = np.ravel(data.Eu_i[n_sweep2]); peak_f = np.nanargmin(peak_E); print("Eu:", '{:.2e} '.format(peak_E[peak_f]))
print(f"[Last  peak - {n_sweep3}]")
peak_E = np.ravel(data.Eu_i[n_sweep3]); peak_f = np.nanargmax(peak_E); print("Eu:", '{:.2e} '.format(peak_E[peak_f]))
peak_E = np.ravel(data.Eu_i[n_sweep3]); peak_f = np.nanargmin(peak_E); print("Eu:", '{:.2e} '.format(peak_E[peak_f]))

# Wave data

In [ ]:
T_HF  = data.T_HF_FPGA[data.n_time//2];  T_RWI = (data.T_RWI_CH1[data.n_time//2] + data.T_RWI_CH2[data.n_time//2])/2
if T_HF  > 199 or T_HF  < -50  or math.isnan(T_HF):
   T_HF  = 25;  data.T_HF_FPGA[:] = math.nan
if T_RWI > 199 or T_RWI < -199 or math.isnan(T_RWI):
   T_RWI = 25;  data.T_RWI_CH1[:] = math.nan;  data.T_RWI_CH2[:] = math.nan

print("Temperature: ", 'HF - {:.1f}'.format(T_HF), '  RWI1 - {:.1f}'.format(T_RWI))

In [ ]:
wave = juice_cal.wave_cal(data, 2, unit_mode, T_HF, T_RWI)
p_max0 = p_raw_max + wave.cf/10;     p_min0 = p_raw_min + wave.cf/10
p_max0 = np.ceil(np.log10( np.nanmax( [np.nanmax(wave.Eu_i), np.nanmax(wave.Ev_i), np.nanmax(wave.Ew_i), np.nanmax(wave.Eu_q), np.nanmax(wave.Ev_q), np.nanmax(wave.Ew_q)] ) )*5)/5+.5

print("unit_mode:", unit_mode, "  conversion factor:", '{:.1f}'.format(wave.cf), 
      "  MAX-min:", '{:.1f}'.format(p_max0), '{:.1f}'.format(p_min0), "  str_unit:", wave.str_unit, 
      "  T-HF & T-RWI:", T_HF, "(", data.T_HF_FPGA[data.n_time//2], ")", T_RWI, "(", data.T_RWI_CH1[data.n_time//2], ")")

In [ ]:
fig = plt.figure(figsize=(16, 11))
ax1 = fig.add_subplot(6, 1, 1);  ax2 = fig.add_subplot(6, 1, 2);  ax3 = fig.add_subplot(6, 1, 3)
ax4 = fig.add_subplot(6, 1, 4);  ax5 = fig.add_subplot(6, 1, 5);  ax6 = fig.add_subplot(6, 1, 6)

ax1.plot(np.ravel(wave.Eu_i[:][:]), '-r', linewidth=.5, label='Eu_i'); ax1.plot(np.ravel(wave.Eu_q[:][:]), ':g', linewidth=.5, label='Eu_q')
ax2.plot(np.ravel(wave.Ev_i[:][:]), '-r', linewidth=.5, label='Ev_i'); ax2.plot(np.ravel(wave.Ev_q[:][:]), ':g', linewidth=.5, label='Ev_q')
ax3.plot(np.ravel(wave.Ew_i[:][:]), '-r', linewidth=.5, label='Ew_i'); ax3.plot(np.ravel(wave.Ew_q[:][:]), ':g', linewidth=.5, label='Ew_q')
ax4.plot(np.ravel(data.frequency),     '-b', linewidth=.5,  label='Frequency')
ax4.plot(np.ravel(data.freq_step*10),  '-g', linewidth=0.8, label='step*10')
ax4.plot(np.ravel(data.freq_width*10), ':b', linewidth=1.0, label='width*10')
ax5.plot(np.ravel(data.T_HF_FPGA),  ':r', label='T (HK-FPGA)');  ax5.plot(np.ravel(data.T_RWI_CH1),     ':b', label='T (RWI1)');  
ax5.plot(np.ravel(data.T_RWI_CH2),  ':g', label='T (RWI2)');     ax5.plot(np.ravel(data.cal_signal*10), '-k', label='CAL-Singal')
ax6.plot(np.ravel(data.epoch[:]), '.')

ax1.set_ylabel('Eu '+wave.str_unit);     ax2.set_ylabel('Ev '+wave.str_unit);  ax3.set_ylabel('Ew '+wave.str_unit);  
ax4.set_ylabel('Frequency [kHz]');  ax5.set_ylabel('T [degC]');      ax6.set_xlabel(str_date)
title_label = '[JUICE/RPWI HF RAW (SID-2)]\n' + data_name;           ax1.set_title(title_label)
ax1.legend(loc='upper right', fontsize=8);  ax2.legend(loc='upper right', fontsize=8);  ax3.legend(loc='upper right', fontsize=8)
ax4.legend(loc='upper right', fontsize=8);  ax5.legend(loc='upper right', fontsize=8)

xlim=[-0.5, len(np.ravel(data.Eu_i[:][:])) -0.5];  ax1.set_xlim(xlim);  ax2.set_xlim(xlim);  ax3.set_xlim(xlim);  ax4.set_xlim(xlim)
xlim=[-0.5, data.n_time                        -0.5];  ax5.set_xlim(xlim);  ax6.set_xlim(xlim)
p_max = np.ceil(np.log10(np.nanmax([np.nanmax(wave.Eu_i), np.nanmax(wave.Eu_q), np.nanmax(wave.Ev_i), np.nanmax(wave.Ev_q), np.nanmax(wave.Ew_i), np.nanmax(wave.Ew_q)]))*5)/5
ylim=[-10**(p_max), 10**(p_max)];  ax1.set_ylim(ylim);  ax2.set_ylim(ylim);  ax3.set_ylim(ylim)
ylim=[f_min0, f_max0];             ax4.set_ylim(ylim)

# Plot
fig.show
if dump_mode == 1:
    png_fname = work_dir+data_name+'_raw.png'
    fig.savefig(png_fname)

# Spectrum Data

In [ ]:
spec = juice_spec.hf_getspec_sid2  (data)                               # Wave -> Spectrum
spec = juice_cal.spec_cal(spec, 2, unit_mode, band_mode, T_HF, T_RWI)   # CAL
spec.EE = spec.EuEu + spec.EvEv + spec.EwEw

spec.epoch = spec.epoch.tolist();         spec.n_time = spec.EuEu.shape[0];  
num_1d    = np.arange(spec.n_time)
freq_1d   = spec.freq  [spec.n_time//2];  n_freq1 = freq_1d.shape[0];    f_min1 = freq_1d[0];  f_max1  = freq_1d[-1]
freq_w_1d = spec.freq_w[spec.n_time//2]

freq_1d_2 = data.frequency2[spec.n_time//2];  n_freq2 = freq_1d_2.shape[0];  f_min2 = freq_1d_2[0];  f_max2  = freq_1d_2[-1]

In [ ]:
p_max1 = p_raw_max + spec.cf/10;  p_min1 = p_raw_min + spec.cf/10
p_max1 = np.ceil(np.log10( np.nanmax( [np.nanmax(spec.EuEu), np.nanmax(spec.EvEv), np.nanmax(spec.EwEw)] ) )*5)/5+.5
p_min1 = p_min1 - (np.log10(freq_w_1d[0])+3)
# p_min1 = np.ceil(np.log10( np.nanmin( [np.nanmin(spec.EuEu), np.nanmin(spec.EvEv), np.nanmin(spec.EwEw)] ) )*5)/5+.5
p_max2 = p_max1 - 1.5;    p_min2 = p_min1 + 1.5;   

print(f"{spec.EuEu.shape}  [frequency] {f_min1} ({f_min1-freq_w_1d[0]/2} - {f_min1+freq_w_1d[0]/2}) - {f_max1} ({f_max1-freq_w_1d[0]/2} - {f_max1+freq_w_1d[0]/2}) kHz  (df = {freq_w_1d[0]} - {freq_w_1d[-1]} kHz)")
print(spec.str_unit, "(unit_mode:", unit_mode, ")  conversion-factor:", '{:.1f}'.format(spec.cf), "[wave:", '{:.1f}'.format(wave.cf),
      "]  T-HF & T-RWI:", T_HF, "(", data.T_HF_FPGA[data.n_time//2], ")", T_RWI, "(", data.T_RWI_CH1[data.n_time//2], ")")
print("[MAX-min]", '{:.1f}'.format(p_max1), "(", '{:.1f}'.format(p_max0), ')  {:.1f}'.format(p_min1), "(", '{:.1f}'.format(p_min0), ")")

In [ ]:
# NAN
if gap_mode == 1 and time_mode == 0:
    for i in range(spec.n_time-1):
        if data.scet[i+1]-data.scet[i] > 60:
            print("[gap]", i, data.epoch[i], i+1, data.epoch[i+1])
            juice_sid2.hf_sid2_spec_nan(data, i);   juice_sid2.hf_sid2_spec_nan(data, i+1)

In [ ]:
n_sweep = spec.n_time//2;  peak_E = np.ravel(spec.EuEu[n_sweep]); peak_f = np.nanargmax(peak_E); 
print(f"[SWEEP - {n_sweep}]  Peak at", '{:.1f}'.format(freq_1d[peak_f]), "kHz  (", peak_f, ")")
print("     EuEu:", '{:+.2e}'.format(np.ravel(spec.EuEu   [n_sweep])[peak_f]), spec.str_unit, "     EvEv:", '{:+.2e}'.format(np.ravel(spec.EvEv   [n_sweep])[peak_f]), spec.str_unit, "   EwEw:", '{:+.2e}'.format(np.ravel(spec.EwEw   [n_sweep])[peak_f]), spec.str_unit)

## Spectrum plot

In [ ]:
EE_2d      = spec.EE.transpose();       EE_med      = np.nanmedian(spec.EE,      axis=0)
EuEu_2d    = spec.EuEu.transpose();     EuEu_med    = np.nanmedian(spec.EuEu,    axis=0)
EvEv_2d    = spec.EvEv.transpose();     EvEv_med    = np.nanmedian(spec.EvEv,    axis=0)
EwEw_2d    = spec.EwEw.transpose();     EwEw_med    = np.nanmedian(spec.EwEw,    axis=0)

In [ ]:
n_sweep1 = 1;  n_sweep2 = spec.n_time//2;  n_sweep3 = spec.n_time-1
p_min = p_min1;  p_max = p_max1
f_min = f_min1;  f_max = f_max1
if f_mode == 0:
    f_min = f_mode_min;  f_max = f_mode_max

fig = plt.figure(figsize=(16, 11))
ax1 = fig.add_subplot(3, 1, 1);  ax2 = fig.add_subplot(3, 1, 2);  ax3 = fig.add_subplot(3, 1, 3)

ax1.plot(freq_1d, spec.EuEu[n_sweep1], '-r', linewidth=.9, label='EuEu 1st')
ax1.plot(freq_1d, spec.EuEu[n_sweep2], '-g', linewidth=.4, label='EuEu mid')
ax1.plot(freq_1d, spec.EuEu[n_sweep3], '-b', linewidth=.2, label='EuEu last')
ax2.plot(freq_1d, spec.EvEv[n_sweep1], '-r', linewidth=.9, label='EvEv 1st')
ax2.plot(freq_1d, spec.EvEv[n_sweep2], '-g', linewidth=.4, label='EvEv mid')
ax2.plot(freq_1d, spec.EvEv[n_sweep3], '-b', linewidth=.2, label='EvEv last')
ax3.plot(freq_1d, spec.EwEw[n_sweep1], '-r', linewidth=.9, label='EwEw 1st')
ax3.plot(freq_1d, spec.EwEw[n_sweep2], '-g', linewidth=.4, label='EwEw mid')
ax3.plot(freq_1d, spec.EwEw[n_sweep3], '-b', linewidth=.2, label='EwEw last')
#
ax1.set_yscale('log');      ax2.set_yscale('log');  ax3.set_yscale('log')
if f_mode == 1:
    ax1.set_xscale('log');  ax2.set_xscale('log');  ax3.set_xscale('log')
ax1.set_ylabel('EuEu '+spec.str_unit);     ax2.set_ylabel('EvEv '+spec.str_unit); ax3.set_ylabel('EwEw '+spec.str_unit);  ax3.set_xlabel('Frequency [kHz]')

date1 = spec.epoch[n_sweep1];  date1 = date1.strftime('1st: %Y/%m/%d %R:%S   ')
date2 = spec.epoch[n_sweep2];  date2 = date2.strftime('Mid: %Y/%m/%d %R:%S   ')
date3 = spec.epoch[n_sweep3];  date3 = date3.strftime('Fin: %Y/%m/%d %R:%S')
title_date = date1 + "  -  " + date2 + "  -  " + date3;   ax1.set_title(title_date)
ax1.legend(loc='upper right', fontsize=8);  ax2.legend(loc='upper right', fontsize=8);  ax3.legend(loc='upper right', fontsize=8)

xlim=[f_min, f_max]
ax1.set_xlim(xlim); ax2.set_xlim(xlim);  ax3.set_xlim(xlim); ax4.set_xlim(xlim)
ylim=[10**(p_min), 10**(p_max)]
ax1.set_ylim(ylim); ax2.set_ylim(ylim);  ax3.set_ylim(ylim); ax4.set_ylim(ylim)

fig.subplots_adjust(hspace=0)
fig.show
if dump_mode == 1:
    png_fname = work_dir+data_name+'_spec.png'
    if f_mode == 1:
        png_fname = work_dir+data_name+'_spec-log.png'
    fig.savefig(png_fname)

In [ ]:
n_sweep1 = 0;  n_sweep2 = spec.n_time//2;  n_sweep3 = spec.n_time-1
p_min = p_min2;  p_max = p_max2
f_min = f_min1;  f_max = f_max1
# f_min = 2000;  f_max = 15000

fig = plt.figure(figsize=(16, 11));  ax1 = fig.add_subplot(1, 1, 1)
# ax1.plot(freq_1d, EE_med,      '-y',  linewidth=4.0, label='EE_med')
ax1.plot(freq_1d, EuEu_med,      '-r',  linewidth=0.9, label='EuEu_med')
ax1.plot(freq_1d, EvEv_med,      '-g',  linewidth=0.4, label='EvEv_med')
ax1.plot(freq_1d, EwEw_med,      '-b',  linewidth=0.2, label='EwEw_med')
ax1.set_yscale('log');      ax2.set_yscale('log');  ax3.set_yscale('log')
if f_mode == 1:
    ax1.set_xscale('log');  ax2.set_xscale('log');  ax3.set_xscale('log')
ax1.set_ylabel('EuEu '+spec.str_unit);     ax2.set_ylabel('EvEv '+spec.str_unit); ax3.set_ylabel('EwEw '+spec.str_unit);  ax3.set_xlabel('Frequency [kHz]')

date1 = spec.epoch[n_sweep1];  date1 = date1.strftime('1st: %Y/%m/%d %R:%S   ')
date2 = spec.epoch[n_sweep2];  date2 = date2.strftime('Mid: %Y/%m/%d %R:%S   ')
date3 = spec.epoch[n_sweep3];  date3 = date3.strftime('Fin: %Y/%m/%d %R:%S')
title_date = date1 + "  -  " + date2 + "  -  " + date3;   ax1.set_title(title_date)
ax1.legend(loc='upper right', fontsize=8);  ax2.legend(loc='upper right', fontsize=8);  ax3.legend(loc='upper right', fontsize=8)

xlim=[f_min, f_max]
ax1.set_xlim(xlim)
ylim=[10**p_min, 10**p_max]; 
ax1.set_ylim(ylim)

fig.subplots_adjust(hspace=0)
fig.show
if dump_mode == 1:
    png_fname = work_dir+data_name+'_spec2.png'
    if f_mode == 1:
        png_fname = work_dir+data_name+'_spec2-log.png'
    fig.savefig(png_fname)

In [ ]:
n_sweep1 = 0;  n_sweep2 = spec.n_time//2;  n_sweep3 = spec.n_time-1
p_min = p_min2;  p_max = p_max2
f_min = 0;       f_max = 5000

fig = plt.figure(figsize=(16, 11))
ax1 = fig.add_subplot(9, 1, 1);  ax2 = fig.add_subplot(9, 1, 2);  ax3 = fig.add_subplot(9, 1, 3);  ax4 = fig.add_subplot(9, 1, 4)
ax5 = fig.add_subplot(9, 1, 5);  ax6 = fig.add_subplot(9, 1, 6);  ax7 = fig.add_subplot(9, 1, 7);  ax8 = fig.add_subplot(9, 1, 8);  ax9 = fig.add_subplot(9, 1, 9)

ax1.plot(freq_1d, spec.EE[n_sweep1], ':r', linewidth=.7, label='EE 1st'); ax1.plot(freq_1d, spec.EE[n_sweep2], ':g', linewidth=.7, label='EE mid')
ax1.plot(freq_1d, spec.EE[n_sweep3], ':b', linewidth=.7, label='EE fin'); ax1.plot(freq_1d, EE_med,            '-k', linewidth=.7, label='EE ave')
ax2.plot(freq_1d, spec.EE[n_sweep1], ':r', linewidth=.7, label='EE 1st'); ax2.plot(freq_1d, spec.EE[n_sweep2], ':g', linewidth=.7, label='EE mid')
ax2.plot(freq_1d, spec.EE[n_sweep3], ':b', linewidth=.7, label='EE fin'); ax2.plot(freq_1d, EE_med,            '-k', linewidth=.7, label='EE ave')
ax3.plot(freq_1d, spec.EE[n_sweep1], ':r', linewidth=.7, label='EE 1st'); ax3.plot(freq_1d, spec.EE[n_sweep2], ':g', linewidth=.7, label='EE mid')
ax3.plot(freq_1d, spec.EE[n_sweep3], ':b', linewidth=.7, label='EE fin'); ax3.plot(freq_1d, EE_med,            '-k', linewidth=.7, label='EE ave')
ax4.plot(freq_1d, spec.EE[n_sweep1], ':r', linewidth=.7, label='EE 1st'); ax4.plot(freq_1d, spec.EE[n_sweep2], ':g', linewidth=.7, label='EE mid')
ax4.plot(freq_1d, spec.EE[n_sweep3], ':b', linewidth=.7, label='EE fin'); ax4.plot(freq_1d, EE_med,            '-k', linewidth=.7, label='EE ave')
ax5.plot(freq_1d, spec.EE[n_sweep1], ':r', linewidth=.7, label='EE 1st'); ax5.plot(freq_1d, spec.EE[n_sweep2], ':g', linewidth=.7, label='EE mid')
ax5.plot(freq_1d, spec.EE[n_sweep3], ':b', linewidth=.7, label='EE fin'); ax5.plot(freq_1d, EE_med,            '-k', linewidth=.7, label='EE ave')
ax6.plot(freq_1d, spec.EE[n_sweep1], ':r', linewidth=.7, label='EE 1st'); ax6.plot(freq_1d, spec.EE[n_sweep2], ':g', linewidth=.7, label='EE mid')
ax6.plot(freq_1d, spec.EE[n_sweep3], ':b', linewidth=.7, label='EE fin'); ax6.plot(freq_1d, EE_med,            '-k', linewidth=.7, label='EE ave')
ax7.plot(freq_1d, spec.EE[n_sweep1], ':r', linewidth=.7, label='EE 1st'); ax7.plot(freq_1d, spec.EE[n_sweep2], ':g', linewidth=.7, label='EE mid')
ax7.plot(freq_1d, spec.EE[n_sweep3], ':b', linewidth=.7, label='EE fin'); ax7.plot(freq_1d, EE_med,            '-k', linewidth=.7, label='EE ave')
ax8.plot(freq_1d, spec.EE[n_sweep1], ':r', linewidth=.7, label='EE 1st'); ax8.plot(freq_1d, spec.EE[n_sweep2], ':g', linewidth=.7, label='EE mid')
ax8.plot(freq_1d, spec.EE[n_sweep3], ':b', linewidth=.7, label='EE fin'); ax8.plot(freq_1d, EE_med,            '-k', linewidth=.7, label='EE ave')
ax9.plot(freq_1d, spec.EE[n_sweep1], ':r', linewidth=.7, label='EE 1st'); ax9.plot(freq_1d, spec.EE[n_sweep2], ':g', linewidth=.7, label='EE mid')
ax9.plot(freq_1d, spec.EE[n_sweep3], ':b', linewidth=.7, label='EE fin'); ax9.plot(freq_1d, EE_med,            '-k', linewidth=.7, label='EE ave')
ax1.set_yscale('log');  ax2.set_yscale('log');  ax3.set_yscale('log');  ax4.set_yscale('log');  ax5.set_yscale('log')
ax6.set_yscale('log');  ax7.set_yscale('log');  ax8.set_yscale('log');  ax9.set_yscale('log')

ax1.set_ylabel(spec.str_unit); ax2.set_ylabel(spec.str_unit); ax3.set_ylabel(spec.str_unit); ax4.set_ylabel(spec.str_unit); ax5.set_ylabel(spec.str_unit)
ax6.set_ylabel(spec.str_unit); ax7.set_ylabel(spec.str_unit); ax8.set_ylabel(spec.str_unit); ax9.set_ylabel(spec.str_unit)
ax9.set_xlabel('Frequency [kHz]')
date1 = spec.epoch[n_sweep1];  date1 = date1.strftime(f'1st {n_sweep1}: %Y/%m/%d %R:%S ')
date2 = spec.epoch[n_sweep2];  date2 = date2.strftime(f'Mid {n_sweep2}: %Y/%m/%d %R:%S ')
date3 = spec.epoch[n_sweep3];  date3 = date3.strftime(f'Fin {n_sweep3}: %Y/%m/%d %R:%S')
title_date = date1 + "  -  " + date2 + "  -  " + date3;  ax1.set_title(title_date)

ax1.legend(loc='upper right', fontsize=8);  ax2.legend(loc='upper right', fontsize=8);  ax3.legend(loc='upper right', fontsize=8)
ax4.legend(loc='upper right', fontsize=8);  ax5.legend(loc='upper right', fontsize=8);  ax6.legend(loc='upper right', fontsize=8)
ax7.legend(loc='upper right', fontsize=8);  ax8.legend(loc='upper right', fontsize=8);  ax9.legend(loc='upper right', fontsize=8)

xlim=[f_min,       f_max];        ax1.set_xlim(xlim); xlim=[f_min+5000,  f_max+5000];   ax2.set_xlim(xlim)
xlim=[f_min+10000, f_max+10000];  ax3.set_xlim(xlim); xlim=[f_min+15000, f_max+15000];  ax4.set_xlim(xlim)
xlim=[f_min+20000, f_max+20000];  ax5.set_xlim(xlim); xlim=[f_min+25000, f_max+25000];  ax6.set_xlim(xlim)
xlim=[f_min+30000, f_max+30000];  ax7.set_xlim(xlim); xlim=[f_min+35000, f_max+35000];  ax8.set_xlim(xlim)
xlim=[f_min+40000, f_max+40000];  ax9.set_xlim(xlim)
ax1.tick_params(labelsize=7); ax2.tick_params(labelsize=7); ax3.tick_params(labelsize=7); ax4.tick_params(labelsize=7); ax5.tick_params(labelsize=7)
ax6.tick_params(labelsize=7); ax7.tick_params(labelsize=7); ax8.tick_params(labelsize=7); ax9.tick_params(labelsize=7)

ylim=[10**p_min, 10**p_max]
ax1.set_ylim(ylim); ax2.set_ylim(ylim); ax3.set_ylim(ylim); ax4.set_ylim(ylim); ax5.set_ylim(ylim); ax6.set_ylim(ylim);  
ax7.set_ylim(ylim); ax8.set_ylim(ylim); ax9.set_ylim(ylim)

fig.subplots_adjust(hspace=0.2)
fig.show
if dump_mode == 1:
    png_fname = work_dir+data_name+'_spec-fine.png'
    fig.savefig(png_fname)

In [ ]:
p_min = p_min2;  p_max = p_max2
f_min = f_min1;  f_max = f_max1
if f_mode == 0:
    f_min = f_mode_min;  f_max = f_mode_max;  print(data.frequency[0][0][0], data.frequency[0][8][0])

fig2d = plt.figure(figsize=[16,11])
if time_mode == 1:
    ax1 = fig2d.add_subplot(4, 1, 1); ax2 = fig2d.add_subplot(4, 1, 2); ax3 = fig2d.add_subplot(4, 1, 3); ax4.set_xlabel(str_date); ax4 = fig2d.add_subplot(4, 1, 4)
else:
    ax1 = fig2d.add_subplot(3, 1, 1); ax2 = fig2d.add_subplot(3, 1, 2); ax3 = fig2d.add_subplot(3, 1, 3); ax3.set_xlabel(str_date)

ax1.set_ylabel('EuEu [kHz]'); ax2.set_ylabel('EvEv [kHz]'); ax3.set_ylabel('EwEw [kHz]')
ax1.set_ylim(f_min, f_max); ax2.set_ylim(f_min, f_max); ax3.set_ylim(f_min, f_max)
if f_mode == 1:
    ax1.set_yscale('log');    ax2.set_yscale('log');        ax3.set_yscale('log')

if time_mode == 1:
    p1 = ax1.pcolormesh(num_1d, freq_1d, EuEu_2d, norm=colors.LogNorm(vmin=10**p_min, vmax=10**p_max), cmap='jet')
    p2 = ax2.pcolormesh(num_1d, freq_1d, EvEv_2d, norm=colors.LogNorm(vmin=10**p_min, vmax=10**p_max), cmap='jet')
    p3 = ax3.pcolormesh(num_1d, freq_1d, EwEw_2d, norm=colors.LogNorm(vmin=10**p_min, vmax=10**p_max), cmap='jet')
    p4 = ax4.plot(np.ravel(spec.epoch[:]), '.')
    pp3 = fig2d.colorbar(p3, ax=ax4, orientation="vertical"); pp3.set_label(spec.str_unit)
else:
    p1 = ax1.pcolormesh(spec.epoch, freq_1d, EuEu_2d, norm=colors.LogNorm(vmin=10**p_min, vmax=10**p_max), cmap='jet')
    p2 = ax2.pcolormesh(spec.epoch, freq_1d, EvEv_2d, norm=colors.LogNorm(vmin=10**p_min, vmax=10**p_max), cmap='jet')
    p3 = ax3.pcolormesh(spec.epoch, freq_1d, EwEw_2d, norm=colors.LogNorm(vmin=10**p_min, vmax=10**p_max), cmap='jet')
pp1 = fig2d.colorbar(p1, ax=ax1, orientation="vertical"); pp1.set_label(spec.str_unit)
pp2 = fig2d.colorbar(p2, ax=ax2, orientation="vertical"); pp2.set_label(spec.str_unit)
pp3 = fig2d.colorbar(p3, ax=ax3, orientation="vertical"); pp3.set_label(spec.str_unit)

print( "(", num_1d[0],")", spec.epoch[0], "-", "(", num_1d[-1],")", spec.epoch[-1] )
if time_mode == 1:
    xlim=[num_1d[0], num_1d[-1]]; ax4.set_xlim(xlim) 
else:
    if t_min0 == 0:
        xlim=[spec.epoch[0], spec.epoch[-1]]
    else:
        xlim=[t_min0, t_max0]
    # E_min = '2024-08-20 18:00:00';  t_min = datetime.datetime.strptime(E_min, "%Y-%m-%d %H:%M:%S");  
    # E_max = '2024-08-21 05:30:00';  t_max = datetime.datetime.strptime(E_max, "%Y-%m-%d %H:%M:%S");  xlim=[t_min, t_max]
    print("==>", xlim)
ax1.set_xlim(xlim);  ax2.set_xlim(xlim); ax3.set_xlim(xlim)

plt.subplots_adjust(hspace=.03)
plt.show
if dump_mode > 0:
    png_fname = work_dir+data_name+'_FT.png'
    if f_mode == 1:
        png_fname = work_dir+data_name+'_FT-log.png'
    fig2d.savefig(png_fname)

# Mask formation for SID-3

#### SID-3 (n_freq_sid3):  freq_1d_sid3, f_step_sid3

In [ ]:
freq_1d_sid3, f_step_sid3, f_width_sid3 = juice_cdf._frequency_sid3(asw_ver);  n_freq_sid3 = len(freq_1d_sid3)

print( f"  sid: {asw_ver}\tn_freq_mask: {n_freq_msk}")
print( f"SID-2: {freq_1d.shape[0]:5d}\tfreq: {freq_1d[0]:.5f} ({freq_1d[0]-freq_w_1d[0]/2:.5f} - {freq_1d[0]+freq_w_1d[0]/2:.5f}) - {freq_1d[-1]:.5f} ({freq_1d[-1]-freq_w_1d[-1]/2:.5f} - {freq_1d[-1]+freq_w_1d[-1]/2:.5f}) [kHz]  \tstep: {freq_w_1d[0]:.5f} - {freq_w_1d[-1]:.5f}")
print( f"SID-3: {freq_1d_sid3.shape[0]:5d}\tfreq: {freq_1d_sid3[0]:.5f} ({freq_1d_sid3[0]-f_step_sid3[0]/2:.5f} - {freq_1d_sid3[0]+f_step_sid3[0]/2:.5f}) - {freq_1d_sid3[-1]:.5f} ({freq_1d_sid3[-1]-f_step_sid3[-1]/2:.5f} - {freq_1d_sid3[-1]+f_step_sid3[-1]/2:.5f}) [kHz]  \tstep: {f_step_sid3[0]:.5f} - {f_step_sid3[-1]:.5f}")

#### Mask (n_freq_msk): freq_1d_msk, freq_w_msk

In [ ]:
# frequency table
freq_1d_msk = np.zeros(n_freq_msk)
for i in range(n_freq_msk):  freq_1d_msk[i] = f_min_msk + f_step_msk * (i + 0.5)      # Central frequency of each channel
freq_w_msk = np.zeros(n_freq_msk);  freq_w_msk[:] = f_step_msk

# flux in MASK table
EE_msk = np.zeros(n_freq_msk)
j0 = 0
for i in range(n_freq_msk):
    if freq_1d_msk[i] >= freq_1d[0]-freq_w_1d[0]/2 and freq_1d_msk[i] < freq_1d[-1]+freq_w_1d[-1]/2:
        for j in range(j0, n_freq_msk):
            if freq_1d_msk[i] >= freq_1d[j]-freq_w_1d[j]/2:
                if freq_1d_msk[i] < freq_1d[j]+freq_w_1d[j]/2:
                    EE_msk[i] = EE_med[j]
                    break
        j0 = j+1
        if j0 >= n_freq_msk:  break

print( f" mask: {freq_1d_msk.shape[0]:5d}\tfreq: {freq_1d_msk[0]:.5f} ({freq_1d_msk[0]-freq_w_msk[0]/2:.5f} - {freq_1d_msk[0]+freq_w_msk[0]/2:.5f}) - {freq_1d_msk[-1]:.5f} ({freq_1d_msk[-1]-freq_w_msk[-1]/2:.5f} - {freq_1d_msk[-1]+freq_w_msk[-1]/2:.5f}) [kHz]  \tstep: {freq_w_msk[0]:.5f} - {freq_w_msk[-1]:.5f}")
print( f"SID-2: {freq_1d.shape[0]    :5d}\tfreq: {freq_1d[0]    :.5f} ({freq_1d[0]    -freq_w_1d[0] /2:.5f} - {freq_1d[0]    +freq_w_1d[0] /2:.5f}) - {freq_1d[-1]    :.5f} ({freq_1d[-1]    -freq_w_1d[-1] /2:.5f} - {freq_1d[-1]    +freq_w_1d[-1] /2:.5f}) [kHz]  \tstep: {freq_w_1d[0] :.5f} - {freq_w_1d[-1] :.5f}")

#### MASK <> SID-3

In [ ]:
# SID3 flux
sid3_EE_med = np.zeros(n_freq_sid3); sid3_EE_masked     = np.zeros(n_freq_sid3)     # SID-3: median / masked
# MASK flux
EE_msk_sid3 = np.zeros(n_freq_msk);  EE_msk_sid3_masked = np.zeros(n_freq_msk)      # MASK:  median / masked
EE_msk_diff = np.zeros(n_freq_msk)                                                  # diff from median

# TABLE to SID-3 (n_freq_sid3, 3):    0 - start MASK-ch, 1 - end MASK-ch,  2 - width MASK-ch
tbl_msk_to_sid3  = juice_cdf._frequency_sid2_to_data(freq_1d_sid3, f_step_sid3, freq_1d_msk, freq_w_msk)
# TABLE to MASK  (n_freq_msk,  4):    0 - start SID-3,   1 - end MASK-ch,  2 - width MASK-ch
tbl_sid3_to_msk  = np.zeros((n_freq_msk, 4), dtype = float);  

# EE median -- thershold value
for j in range(n_freq_sid3):
    tbl = tbl_msk_to_sid3[j]                            # SID-3:   0 - start MASK-ch, 1 - end MASK-ch,  2 - width MASK-ch
    sid3_EE_med[j]  = np.nanmedian( EE_msk[tbl[0]:tbl[1]+1] )      # **** 10 ** np.median( np.log10(EE_msk[tbl[0]:tbl[1]+1]) ) ****
    EE_msk_sid3[tbl[0]:tbl[1]+1] = sid3_EE_med[j]
    tbl_sid3_to_msk [tbl[0]:tbl[1]+1, 0:3] = tbl    # MASK:    0 - start MASK-ch, 1 - end MASK-ch,  2 - width MASK-ch
    tbl_sid3_to_msk [tbl[0]:tbl[1]+1, 3]   = j      # MASK:    3 - SID-3 channel

In [ ]:
# if filter_mode > 0:
p_min = p_min2;  p_max = p_max2
f_min = f_min1;  f_max = f_max1
f_min = freq_1d_msk[0]
# f_min = 100;     f_max = 500
lw = 1;  ms = 5
if f_mode == 0:
    f_min = f_mode_min;  f_max = f_mode_max

fig = plt.figure(figsize=(14, 11));  ax1 = fig.add_subplot(1, 1, 1)
ax1.plot(freq_1d,      EE_med,          '.y', linewidth=lw,   ms=ms*4, label='EE_raw   (SID2)')
ax1.plot(freq_1d_msk,  EE_msk,          '-b', linewidth=lw,   ms=ms,   label='EE_mask  (SID2)')
ax1.plot(freq_1d_msk,  EE_msk_sid3,     '.g', linewidth=lw,   ms=ms,   label='EE_med   (SID2)')
ax1.plot(freq_1d_sid3, sid3_EE_med,     '.r', linewidth=lw,   ms=ms*2, label='EE_thres (SID3-res)')
ax1.set_yscale('log')
if f_mode == 1:                   ax1.set_xscale('log')
xlim=[f_min, f_max];              ax1.set_xlim(xlim)
ylim=[10**(p_min), 10**(p_max)];  ax1.set_ylim(ylim)

ax1.set_ylabel('EE '+spec.str_unit)
date1 = spec.epoch[n_sweep1];  date1 = date1.strftime('1st: %Y/%m/%d %R:%S   ')
date2 = spec.epoch[n_sweep2];  date2 = date2.strftime('Mid: %Y/%m/%d %R:%S   ')
date3 = spec.epoch[n_sweep3];  date3 = date3.strftime('Fin: %Y/%m/%d %R:%S')
title_date = date1 + "  -  " + date2 + "  -  " + date3
ax1.set_title(title_date);   ax1.legend(loc='upper right', fontsize=8)
plt.subplots_adjust(hspace=.03)
fig.show

#### MASK formation

In [ ]:
# EE_med_sid3[tbl[1]:n_freq1] = math.nan;  EE_ave_sid3[tbl[1]:n_freq1] = math.nan;  EE_msk_sid3[tbl[1]:n_freq1] = math.nan
for i in range(n_freq_msk):
    if EE_msk[i] > 0 and EE_msk_sid3[i] > 0:
        EE_msk_diff[i] = 10 * np.log10(EE_msk[i]) - 10 * np.log10(EE_msk_sid3[i]) - sid3_diff
    else:
        EE_msk_diff[i] =                                                          - sid3_diff

# Mask table
mask_list = np.zeros((n_freq_msk, 4), dtype = float)    # MASK table:  0 - f_min, 1 - f_max, 2 - mask, 3 - non-mask rate
for i in range(n_freq_msk):
    mask_list[i, 0] = freq_1d_msk[i] - 2.3125/2         # f_min
    mask_list[i, 1] = freq_1d_msk[i] + 2.3125/2         # f_max
    mask_list[i, 2] = 0                                 # mask
    if EE_msk_diff[i] >= 0: mask_list[i, 2] = 1

for i in range(n_freq_sid3):
    num_mask = 0
    tbl = tbl_msk_to_sid3[i]                            # SID-3 table:  0 - start ch, 1 - end ch, 2 - width ch
    for j in range (tbl[0], tbl[1]+1):
        if mask_list[j, 2] == 0:  num_mask += 1.0
    for j in range (tbl[0], tbl[1]+1):
        mask_list[j, 3] = num_mask / tbl[2]

#### WRITE MASK table

In [ ]:
print("   Ch   f_c[kHz]   f_b[kHz] mask ratio[%] SID3_ch")
for i in range(n_freq_msk):
    print(f"{i+1:5d} {mask_list[i, 0]:10.4f} {mask_list[i, 1]:10.4f}    {mask_list[i, 2]:.0f}     {mask_list[i, 3]*100:3.0f}  {tbl_sid3_to_msk[i, 3]:3.0f}")

# Write Mask
if filter_mode > 0:
    if (asw_ver < 3):  mask_file = './lib/hf_sid3_mask_80k.csv';  
    else:              mask_file = './lib/hf_sid3_mask_20k.csv';  
    with open(mask_file, 'w') as f:
        writer = csv.writer(f)
        for i in range(n_freq_msk):
            # MASK table:  0 - f_min, 1 - f_max, 2 - mask
            writer.writerow([ i+1, mask_list[i, 0], mask_list[i, 1], np.int16(mask_list[i, 2]), mask_list[i, 3]*100, tbl_sid3_to_msk[i, 3]])

#### WRITE Frequency table

In [ ]:
# SID-3 Frequeny table write ---- ch, frequency, num of SID-2 ch, masked ch, masked ratio, f_min, f_max
print(" Ch   f_c[kHz]   f_b[kHz]   f_e[kHz]  ch_s  ch_e ch_w mask ratio[%]")
for i in range(n_freq_sid3):
    i0 = tbl_msk_to_sid3[i][0];  i1 = tbl_msk_to_sid3[i][1];  i2 = tbl_msk_to_sid3[i][2]
    num = 0
    for j in range (i0, i1+1):
        if mask_list[j, 2] == 0:  num += 1.0
    print(f"{i+1:3d} {freq_1d_sid3[i]:10.4f} {freq_1d_msk[i0]-freq_w_msk[i0]/2:10.4f} {freq_1d_msk[i1]+freq_w_msk[i1]/2:10.4f} {tbl_msk_to_sid3[i][0]:5d} {tbl_msk_to_sid3[i][1]:5d}  {tbl_msk_to_sid3[i][2]:3d}  {num:3.0f} {num/i2*100:3.0f}")
    
if filter_sid3 > 0:
    if (asw_ver < 3):  freq_file = './lib/hf_sid3_freq_80k.csv';  
    else:              freq_file = './lib/hf_sid3_freq_20k.csv';  
    with open(freq_file, 'w') as f:
        writer = csv.writer(f)
        print(["Ch", "fc_kHz", "fb_kHz", "fe_kHz", "ch_s", "ch_e", "ch_w", "mask", "ratio_%"])
        writer.writerow(["Ch", "fc_kHz", "fb_kHz", "fe_kHz", "ch_s", "ch_e", "ch_w", "mask", "ratio_%"])
        for i in range(n_freq_sid3):
            i0 = tbl_msk_to_sid3[i][0];  i1 = tbl_msk_to_sid3[i][1]
            writer.writerow([ i+1, freq_1d_sid3[i], freq_1d_msk[i0]-freq_w_msk[i0]/2, freq_1d_msk[i1]+freq_w_msk[i1]/2, tbl_msk_to_sid3[i][2] ])   
         

# SID-3 frequencies

In [ ]:
freq_1d_sid3_ASW20, f_step_sid3_ASW20, f_width_sid3_ASW20 = juice_cdf._frequency_sid3(2.0)
freq_1d_sid3_ASW30, f_step_sid3_ASW30, f_width_sid3_ASW30 = juice_cdf._frequency_sid3(3.0)
freq_1d_sid3_ASW31, f_step_sid3_ASW31, f_width_sid3_ASW31 = juice_cdf._frequency_sid3(3.1)
freq_1d_sid3_ASW32, f_step_sid3_ASW32, f_width_sid3_ASW32 = juice_cdf._frequency_sid3(3.2)

fig = plt.figure(figsize=(14, 11))
ax1 = fig.add_subplot(2, 1, 1)
ax1.plot(freq_1d_sid3_ASW20,   ':k', linewidth=1.0, label='ASW2.0')
ax1.plot(freq_1d_sid3_ASW30,   ':r', linewidth=1.0, label='ASW3.0')
ax1.plot(freq_1d_sid3_ASW31,   '-g', linewidth=1.6, label='ASW3.1')
ax1.plot(freq_1d_sid3_ASW32,   '-b', linewidth=0.6, label='ASW3.2')
ax2 = fig.add_subplot(2, 1, 2)
ax2.plot(f_step_sid3_ASW20,   ':k', linewidth=1.0, label='ASW2.0')
ax2.plot(f_step_sid3_ASW30,   ':r', linewidth=1.0, label='ASW3.0')
ax2.plot(f_step_sid3_ASW31,   '-g', linewidth=1.6, label='ASW3.1')
ax2.plot(f_step_sid3_ASW32,   '-b', linewidth=0.6, label='ASW3.2')
ax2.legend(loc='upper left', fontsize=8);  ax2.set_yscale('log')
fig.subplots_adjust(hspace=0);  fig.show
ax1.legend(loc='upper left', fontsize=8);  ax1.set_yscale('log')
fig.subplots_adjust(hspace=0);  fig.show
png_fname = work_dir+data_name+'_F_ASW3.0.png';  fig.savefig(png_fname)

# Mask Check

#### Mask read

In [ ]:
if (asw_ver < 3):  mask_file = './lib/hf_sid3_mask_80k.csv';  
else:              mask_file = './lib/hf_sid3_mask_20k.csv';  

# Read Mask
mask_list = np.zeros((n_freq_msk, 3), dtype = float)
with open(mask_file, 'r') as f:
    reader = csv.reader(f);  i = 0
    for row in reader:
        mask_list[i, 0] = row[1];  mask_list[i, 1] = row[2];  mask_list[i, 2] = row[3];  i = i+1
print("mask read: ", mask_list.shape, i, n_freq_msk, mask_list[0], mask_list[-1] )
print("   Ch   f_c[kHz]   f_b[kHz] mask ratio[%] SID3_ch")
for i in range(n_freq_msk):
   print(f"{i+1:5d} {mask_list[i, 0]:10.4f} {mask_list[i, 1]:10.4f}    {mask_list[i, 2]:.0f}")

# SID-3 Frequency table - ASW3.0

In [ ]:
asw30_ver = 3.0
freq_1d_sid3, f_step_sid3, f_width_sid3 = juice_cdf._frequency_sid3(asw30_ver);  n_freq_sid3 = len(freq_1d_sid3)

print( f"  sid: {asw30_ver}\tn_freq_mask: {n_freq_msk}")
print( f"SID-2: {freq_1d.shape[0]:5d}\tfreq: {freq_1d[0]:.5f} ({freq_1d[0]-freq_w_1d[0]/2:.5f} - {freq_1d[0]+freq_w_1d[0]/2:.5f}) - {freq_1d[-1]:.5f} ({freq_1d[-1]-freq_w_1d[-1]/2:.5f} - {freq_1d[-1]+freq_w_1d[-1]/2:.5f}) [kHz]  \tstep: {freq_w_1d[0]:.5f} - {freq_w_1d[-1]:.5f}")
print( f"SID-3: {freq_1d_sid3.shape[0]:5d}\tfreq: {freq_1d_sid3[0]:.5f} ({freq_1d_sid3[0]-f_step_sid3[0]/2:.5f} - {freq_1d_sid3[0]+f_step_sid3[0]/2:.5f}) - {freq_1d_sid3[-1]:.5f} ({freq_1d_sid3[-1]-f_step_sid3[-1]/2:.5f} - {freq_1d_sid3[-1]+f_step_sid3[-1]/2:.5f}) [kHz]  \tstep: {f_step_sid3[0]:.5f} - {f_step_sid3[-1]:.5f}")

In [ ]:
# TABLE to SID-3 (n_freq_sid3, 3):    0 - start MASK-ch, 1 - end MASK-ch,  2 - width MASK-ch
tbl_msk_to_sid3  = juice_cdf._frequency_sid2_to_data(freq_1d_sid3, f_step_sid3, freq_1d_msk, freq_w_msk)
for i in range(n_freq_sid3):
    i0 = tbl_msk_to_sid3[i][0];  i1 = tbl_msk_to_sid3[i][1];  i2 = tbl_msk_to_sid3[i][2]
    num = 0
    for j in range (i0, i1+1):
        if mask_list[j, 2] == 0:  num += 1.0

In [ ]:
# masked SID-2
EE_masked = copy.deepcopy(EE_med)
j0        = 0
for i in range(n_freq_msk):
    if mask_list[i, 2] > 0:
        for j in range(j0, n_freq1):
            if freq_1d[j] >  mask_list[i, 1]:
                break
            if freq_1d[j] >= mask_list[i, 0] and freq_1d[j] < mask_list[i, 1]:
                EE_masked[j] = math.nan
        j0 = j

# masked SID-3
sid3_EE_med = np.zeros(n_freq_sid3); sid3_EE_ave = np.zeros(n_freq_sid3); sid3_EE_masked = np.zeros(n_freq_sid3)     # median / masked
tbl_sid2_to_sid3  = juice_cdf._frequency_sid2_to_data(freq_1d_sid3, f_step_sid3, freq_1d, freq_w_1d)
for j in range(n_freq_sid3):
    tbl = tbl_sid2_to_sid3[j]                        # 0 - start ch, 1 - end ch, 2 - width ch
    sid3_EE_med   [j] = np.nanmedian( EE_med   [tbl[0]:tbl[1]+1] )
    sid3_EE_ave   [j] = np.nanmean  ( EE_med   [tbl[0]:tbl[1]+1] )
    sid3_EE_masked[j] = np.nanmean  ( EE_masked[tbl[0]:tbl[1]+1] )
    if np.sum(np.isnan(EE_masked[tbl[0]:tbl[1]+1]))/tbl[2] >= 0.5:
        print(j, freq_1d_sid3[j], tbl[2], np.sum(np.isnan(EE_masked[tbl[0]:tbl[1]+1])), np.sum(np.isnan(EE_masked[tbl[0]:tbl[1]+1]))/tbl[2]*100)

In [ ]:
p_min = p_min2;  p_max = p_max2
f_min = f_min1;  f_max = f_max1
if f_mode == 0:
    f_min = f_mode_min;  f_max = f_mode_max
lw = 1;  ms = 3

fig = plt.figure(figsize=(14, 11));  ax1 = fig.add_subplot(1, 1, 1)
ax1.plot(freq_1d,       EE_med,          '.y', linewidth=lw,   ms=ms,   label='EE_raw   (SID2)')
ax1.plot(freq_1d,       EE_masked,       '.k', linewidth=lw,   ms=ms,   label='EE_mask  (SID2)')
ax1.plot(freq_1d_sid3,  sid3_EE_med,     '-g', linewidth=lw*2, ms=ms,   label='EE_med   (SID3)')
ax1.plot(freq_1d_sid3,  sid3_EE_ave,     '-b', linewidth=lw,   ms=ms,   label='EE_ave   (SID3)')
ax1.plot(freq_1d_sid3,  sid3_EE_masked,  '-r', linewidth=lw*4, ms=ms*2, label='EE_mask  (SID3)')
ax1.set_yscale('log')
if f_mode == 1:                   ax1.set_xscale('log')
xlim=[f_min, f_max];              ax1.set_xlim(xlim)
ylim=[10**(p_min), 10**(p_max)];  ax1.set_ylim(ylim)

ax1.set_ylabel('EE '+spec.str_unit)
date1 = spec.epoch[n_sweep1];  date1 = date1.strftime('1st: %Y/%m/%d %R:%S   ')
date2 = spec.epoch[n_sweep2];  date2 = date2.strftime('Mid: %Y/%m/%d %R:%S   ')
date3 = spec.epoch[n_sweep3];  date3 = date3.strftime('Fin: %Y/%m/%d %R:%S')
title_date = "[ASW3.0] " + date1 + "  -  " + date2 + "  -  " + date3
ax1.set_title(title_date);   ax1.legend(loc='upper right', fontsize=8)
plt.subplots_adjust(hspace=.03)
fig.show
png_fname = work_dir+data_name+'_W_ASW3.0.png';  fig.savefig(png_fname)

In [ ]:
p_min = p_min2;   p_max = p_max2
f_min = 0;        f_max = 5000

fig = plt.figure(figsize=(14, 20))
ax1 = fig.add_subplot(9, 1, 1);  ax2 = fig.add_subplot(9, 1, 2);  ax3 = fig.add_subplot(9, 1, 3);  ax4 = fig.add_subplot(9, 1, 4)
ax5 = fig.add_subplot(9, 1, 5);  ax6 = fig.add_subplot(9, 1, 6);  ax7 = fig.add_subplot(9, 1, 7);  ax8 = fig.add_subplot(9, 1, 8)
ax9 = fig.add_subplot(9, 1, 9)

ax1.plot(freq_1d,      EE_med,          '-y', linewidth=lw, ms=ms*4, label='EE_raw   (SID2)')
ax1.plot(freq_1d,      EE_masked,       '.b', linewidth=lw, ms=ms,   label='EE_mask  (SID2)')
ax1.plot(freq_1d_msk,  mask_list[:, 2]*1E2, '.r', linewidth=.3,  label='mask pattern')
ax1.plot(freq_1d_sid3, sid3_EE_med,     '-g', linewidth=lw*2, ms=ms,   label='EE_med   (SID3)')
ax1.plot(freq_1d_sid3, sid3_EE_ave,     '-b', linewidth=lw,   ms=ms,   label='EE_ave   (SID3)')
ax1.plot(freq_1d_sid3, sid3_EE_masked,  '-r', linewidth=lw*4, ms=ms*2, label='EE_mask  (SID3)')
#
ax2.plot(freq_1d,      EE_med,          '-y', linewidth=lw, ms=ms*4, label='EE_raw   (SID2)')
ax2.plot(freq_1d,      EE_masked,       '.b', linewidth=lw, ms=ms, label='EE_mask  (SID2)')
ax2.plot(freq_1d_msk,  mask_list[:, 2]*1E2, '.r', linewidth=.3,  label='mask pattern')
ax2.plot(freq_1d_sid3, sid3_EE_med,     '-g', linewidth=lw*2, ms=ms,   label='EE_med   (SID3)')
ax2.plot(freq_1d_sid3, sid3_EE_ave,     '-b', linewidth=lw,   ms=ms,   label='EE_ave   (SID3)')
ax2.plot(freq_1d_sid3, sid3_EE_masked,  '-r', linewidth=lw*4, ms=ms*2, label='EE_mask  (SID3)')
#
ax3.plot(freq_1d,      EE_med,          '-y', linewidth=lw, ms=ms*4, label='EE_raw   (SID2)')
ax3.plot(freq_1d,      EE_masked,       '.b', linewidth=lw, ms=ms, label='EE_mask  (SID2)')
ax3.plot(freq_1d_msk,  mask_list[:, 2]*1E2, '.r', linewidth=.3,  label='mask pattern')
ax3.plot(freq_1d_sid3, sid3_EE_med,     '-g', linewidth=lw*2, ms=ms,   label='EE_med   (SID3)')
ax3.plot(freq_1d_sid3, sid3_EE_ave,     '-b', linewidth=lw,   ms=ms,   label='EE_ave   (SID3)')
ax3.plot(freq_1d_sid3, sid3_EE_masked,  '-r', linewidth=lw*4, ms=ms*2, label='EE_mask  (SID3)')
#
ax4.plot(freq_1d,      EE_med,          '-y', linewidth=lw, ms=ms*4, label='EE_raw   (SID2)')
ax4.plot(freq_1d,      EE_masked,       '.b', linewidth=lw, ms=ms, label='EE_mask  (SID2)')
ax4.plot(freq_1d_msk,  mask_list[:, 2]*1E2, '.r', linewidth=.3,  label='mask pattern')
ax4.plot(freq_1d_sid3, sid3_EE_med,     '-g', linewidth=lw*2, ms=ms,   label='EE_med   (SID3)')
ax4.plot(freq_1d_sid3, sid3_EE_ave,     '-b', linewidth=lw,   ms=ms,   label='EE_ave   (SID3)')
ax4.plot(freq_1d_sid3, sid3_EE_masked,  '-r', linewidth=lw*4, ms=ms*2, label='EE_mask  (SID3)')
#
ax5.plot(freq_1d,      EE_med,          '-y', linewidth=lw, ms=ms*4, label='EE_raw   (SID2)')
ax5.plot(freq_1d,      EE_masked,       '.b', linewidth=lw, ms=ms, label='EE_mask  (SID2)')
ax5.plot(freq_1d_msk,  mask_list[:, 2]*1E2 ,   '.r', linewidth=.3,  label='mask pattern')
ax5.plot(freq_1d_sid3, sid3_EE_med,     '-g', linewidth=lw*2, ms=ms,   label='EE_med   (SID3)')
ax5.plot(freq_1d_sid3, sid3_EE_ave,     '-b', linewidth=lw,   ms=ms,   label='EE_ave   (SID3)')
ax5.plot(freq_1d_sid3, sid3_EE_masked,  '-r', linewidth=lw*4, ms=ms*2, label='EE_mask  (SID3)')
#
ax6.plot(freq_1d,      EE_med,          '-y', linewidth=lw, ms=ms*4, label='EE_raw   (SID2)')
ax6.plot(freq_1d,      EE_masked,       '.b', linewidth=lw, ms=ms, label='EE_mask  (SID2)')
ax6.plot(freq_1d_msk,  mask_list[:, 2]*1E2,   '.r', linewidth=.3,  label='mask pattern')
ax6.plot(freq_1d_sid3, sid3_EE_med,     '-g', linewidth=lw*2, ms=ms,   label='EE_med   (SID3)')
ax6.plot(freq_1d_sid3, sid3_EE_ave,     '-b', linewidth=lw,   ms=ms,   label='EE_ave   (SID3)')
ax6.plot(freq_1d_sid3, sid3_EE_masked,  '-r', linewidth=lw*4, ms=ms*2, label='EE_mask  (SID3)')
#
ax7.plot(freq_1d,      EE_med,          '-y', linewidth=lw, ms=ms*4, label='EE_raw   (SID2)')
ax7.plot(freq_1d,      EE_masked,       '.b', linewidth=lw, ms=ms, label='EE_mask  (SID2)')
ax7.plot(freq_1d_msk,  mask_list[:, 2]*1E2,   '.r', linewidth=.3,  label='mask pattern')
ax7.plot(freq_1d_sid3, sid3_EE_med,     '-g', linewidth=lw*2, ms=ms,   label='EE_med   (SID3)')
ax7.plot(freq_1d_sid3, sid3_EE_ave,     '-b', linewidth=lw,   ms=ms,   label='EE_ave   (SID3)')
ax7.plot(freq_1d_sid3, sid3_EE_masked,  '-r', linewidth=lw*4, ms=ms*2, label='EE_mask  (SID3)')
#
ax8.plot(freq_1d,      EE_med,          '-y', linewidth=lw, ms=ms*4, label='EE_raw   (SID2)')
ax8.plot(freq_1d,      EE_masked,       '.b', linewidth=lw, ms=ms, label='EE_mask  (SID2)')
ax8.plot(freq_1d_msk,  mask_list[:, 2]*1E2,   '.r', linewidth=.3,  label='mask pattern')
ax8.plot(freq_1d_sid3, sid3_EE_med,     '-g', linewidth=lw*2, ms=ms,   label='EE_med   (SID3)')
ax8.plot(freq_1d_sid3, sid3_EE_ave,     '-b', linewidth=lw,   ms=ms,   label='EE_ave   (SID3)')
ax8.plot(freq_1d_sid3, sid3_EE_masked,  '-r', linewidth=lw*4, ms=ms*2, label='EE_mask  (SID3)')
#
ax9.plot(freq_1d,      EE_med,          '-y', linewidth=lw, ms=ms*4, label='EE_raw   (SID2)')
ax9.plot(freq_1d,      EE_masked,       '.b', linewidth=lw, ms=ms, label='EE_mask  (SID2)')
ax9.plot(freq_1d_msk,  mask_list[:, 2]*1E2,   '.r', linewidth=.3,  label='mask pattern')
ax9.plot(freq_1d_sid3, sid3_EE_med,     '-g', linewidth=lw*2, ms=ms,   label='EE_med   (SID3)')
ax9.plot(freq_1d_sid3, sid3_EE_ave,     '-b', linewidth=lw,   ms=ms,   label='EE_ave   (SID3)')
ax9.plot(freq_1d_sid3, sid3_EE_masked,  '-r', linewidth=lw*4, ms=ms*2, label='EE_mask  (SID3)')
#
ax1.set_yscale('log');  ax2.set_yscale('log');  ax3.set_yscale('log');  ax4.set_yscale('log');  ax5.set_yscale('log')
ax6.set_yscale('log');  ax7.set_yscale('log');  ax8.set_yscale('log');  ax9.set_yscale('log')

# Label
ax1.set_ylabel(spec.str_unit); ax2.set_ylabel(spec.str_unit); ax3.set_ylabel(spec.str_unit); ax4.set_ylabel(spec.str_unit); ax5.set_ylabel(spec.str_unit)
ax6.set_ylabel(spec.str_unit); ax7.set_ylabel(spec.str_unit); ax8.set_ylabel(spec.str_unit); ax9.set_ylabel(spec.str_unit)
ax9.set_xlabel('Frequency [kHz]')
#
date1 = spec.epoch[n_sweep1];  date1 = date1.strftime(f'1st {n_sweep1}: %Y/%m/%d %R:%S   ')
date2 = spec.epoch[n_sweep2];  date2 = date2.strftime(f'Mid {n_sweep2}: %Y/%m/%d %R:%S   ')
date3 = spec.epoch[n_sweep3];  date3 = date3.strftime(f'Fin {n_sweep3}: %Y/%m/%d %R:%S')
title_date = "[ASW3.0] " + date1 + "  -  " + date2 + "  -  " + date3
ax1.set_title(title_date)

ax1.legend(loc='upper right', fontsize=8);  ax2.legend(loc='upper right', fontsize=8);  ax3.legend(loc='upper right', fontsize=8)
ax4.legend(loc='upper right', fontsize=8);  ax5.legend(loc='upper right', fontsize=8);  ax6.legend(loc='upper right', fontsize=8)
ax7.legend(loc='upper right', fontsize=8);  ax8.legend(loc='upper right', fontsize=8);  ax9.legend(loc='upper right', fontsize=8)

# range: X-axis
xlim=[f_min,       f_max];        ax1.set_xlim(xlim);  xlim=[f_min+5000,  f_max+5000];   ax2.set_xlim(xlim);  
xlim=[f_min+10000, f_max+10000];  ax3.set_xlim(xlim);  xlim=[f_min+15000, f_max+15000];  ax4.set_xlim(xlim)
xlim=[f_min+20000, f_max+20000];  ax5.set_xlim(xlim);  xlim=[f_min+25000, f_max+25000];  ax6.set_xlim(xlim)
xlim=[f_min+30000, f_max+30000];  ax7.set_xlim(xlim);  xlim=[f_min+35000, f_max+35000];  ax8.set_xlim(xlim)
xlim=[f_min+40000, f_max+40000];  ax9.set_xlim(xlim)
ax1.tick_params(labelsize=7); ax2.tick_params(labelsize=7); ax3.tick_params(labelsize=7); ax4.tick_params(labelsize=7); ax5.tick_params(labelsize=7)
ax6.tick_params(labelsize=7); ax7.tick_params(labelsize=7); ax8.tick_params(labelsize=7); ax9.tick_params(labelsize=7)

ylim=[10**p_min, 10**p_max]
ax1.set_ylim(ylim); ax2.set_ylim(ylim);  ax3.set_ylim(ylim); ax4.set_ylim(ylim); ax5.set_ylim(ylim); ax6.set_ylim(ylim);  
ax7.set_ylim(ylim); ax8.set_ylim(ylim);  ax9.set_ylim(ylim)

fig.subplots_adjust(hspace=0)
fig.show
png_fname = work_dir+data_name+'_N_ASW3.0.png';  fig.savefig(png_fname)


# SID-3 Frequency table - ASW3.1

In [ ]:
asw31_ver = 3.1
freq_1d_sid3, f_step_sid3, f_width_sid3 = juice_cdf._frequency_sid3(asw31_ver);  n_freq_sid3 = len(freq_1d_sid3)

print( f"  sid: {asw31_ver}\tn_freq_mask: {n_freq_msk}")
print( f"SID-2: {freq_1d.shape[0]:5d}\tfreq: {freq_1d[0]:.5f} ({freq_1d[0]-freq_w_1d[0]/2:.5f} - {freq_1d[0]+freq_w_1d[0]/2:.5f}) - {freq_1d[-1]:.5f} ({freq_1d[-1]-freq_w_1d[-1]/2:.5f} - {freq_1d[-1]+freq_w_1d[-1]/2:.5f}) [kHz]  \tstep: {freq_w_1d[0]:.5f} - {freq_w_1d[-1]:.5f}")
print( f"SID-3: {freq_1d_sid3.shape[0]:5d}\tfreq: {freq_1d_sid3[0]:.5f} ({freq_1d_sid3[0]-f_step_sid3[0]/2:.5f} - {freq_1d_sid3[0]+f_step_sid3[0]/2:.5f}) - {freq_1d_sid3[-1]:.5f} ({freq_1d_sid3[-1]-f_step_sid3[-1]/2:.5f} - {freq_1d_sid3[-1]+f_step_sid3[-1]/2:.5f}) [kHz]  \tstep: {f_step_sid3[0]:.5f} - {f_step_sid3[-1]:.5f}")

In [ ]:
# TABLE to SID-3 (n_freq_sid3, 3):    0 - start MASK-ch, 1 - end MASK-ch,  2 - width MASK-ch
tbl_msk_to_sid3  = juice_cdf._frequency_sid2_to_data(freq_1d_sid3, f_step_sid3, freq_1d_msk, freq_w_msk)
for i in range(n_freq_sid3):
    i0 = tbl_msk_to_sid3[i][0];  i1 = tbl_msk_to_sid3[i][1];  i2 = tbl_msk_to_sid3[i][2]
    num = 0
    for j in range (i0, i1+1):
        if mask_list[j, 2] == 0:  num += 1.0

In [ ]:
# masked SID-2
EE_masked = copy.deepcopy(EE_med)
j0        = 0
for i in range(n_freq_msk):
    if mask_list[i, 2] > 0:
        for j in range(j0, n_freq1):
            if freq_1d[j] >  mask_list[i, 1]:
                break
            if freq_1d[j] >= mask_list[i, 0] and freq_1d[j] < mask_list[i, 1]:
                EE_masked[j] = math.nan
        j0 = j

# masked SID-3
sid3_EE_med = np.zeros(n_freq_sid3); sid3_EE_ave = np.zeros(n_freq_sid3); sid3_EE_masked = np.zeros(n_freq_sid3)     # median / masked
tbl_sid2_to_sid3  = juice_cdf._frequency_sid2_to_data(freq_1d_sid3, f_step_sid3, freq_1d, freq_w_1d)
for j in range(n_freq_sid3):
    tbl = tbl_sid2_to_sid3[j]                        # 0 - start ch, 1 - end ch, 2 - width ch
    sid3_EE_med   [j] = np.nanmedian( EE_med   [tbl[0]:tbl[1]+1] )
    sid3_EE_ave   [j] = np.nanmean  ( EE_med   [tbl[0]:tbl[1]+1] )
    sid3_EE_masked[j] = np.nanmean  ( EE_masked[tbl[0]:tbl[1]+1] )
    if np.sum(np.isnan(EE_masked[tbl[0]:tbl[1]+1]))/tbl[2] >= 0.5:
        print(j, freq_1d_sid3[j], tbl[2], np.sum(np.isnan(EE_masked[tbl[0]:tbl[1]+1])), np.sum(np.isnan(EE_masked[tbl[0]:tbl[1]+1]))/tbl[2]*100)

In [ ]:
p_min = p_min2;  p_max = p_max2
f_min = f_min1;  f_max = f_max1
if f_mode == 0:
    f_min = f_mode_min;  f_max = f_mode_max
lw = 1;  ms = 3

fig = plt.figure(figsize=(14, 11));  ax1 = fig.add_subplot(1, 1, 1)
ax1.plot(freq_1d,       EE_med,          '.y', linewidth=lw,   ms=ms,   label='EE_raw   (SID2)')
ax1.plot(freq_1d,       EE_masked,       '.k', linewidth=lw,   ms=ms,   label='EE_mask  (SID2)')
ax1.plot(freq_1d_sid3,  sid3_EE_med,     '-g', linewidth=lw*2, ms=ms,   label='EE_med   (SID3)')
ax1.plot(freq_1d_sid3,  sid3_EE_ave,     '-b', linewidth=lw,   ms=ms,   label='EE_ave   (SID3)')
ax1.plot(freq_1d_sid3,  sid3_EE_masked,  '-r', linewidth=lw*4, ms=ms*2, label='EE_mask  (SID3)')
ax1.set_yscale('log')
if f_mode == 1:                   ax1.set_xscale('log')
xlim=[f_min, f_max];              ax1.set_xlim(xlim)
ylim=[10**(p_min), 10**(p_max)];  ax1.set_ylim(ylim)

ax1.set_ylabel('EE '+spec.str_unit)
date1 = spec.epoch[n_sweep1];  date1 = date1.strftime('1st: %Y/%m/%d %R:%S   ')
date2 = spec.epoch[n_sweep2];  date2 = date2.strftime('Mid: %Y/%m/%d %R:%S   ')
date3 = spec.epoch[n_sweep3];  date3 = date3.strftime('Fin: %Y/%m/%d %R:%S')
title_date = "[ASW3.1] " + date1 + "  -  " + date2 + "  -  " + date3
ax1.set_title(title_date);   ax1.legend(loc='upper right', fontsize=8)
plt.subplots_adjust(hspace=.03)
fig.show
png_fname = work_dir+data_name+'_W_ASW3.1.png';  fig.savefig(png_fname)

In [ ]:
p_min = p_min2;   p_max = p_max2
f_min = 0;        f_max = 5000

fig = plt.figure(figsize=(14, 20))
ax1 = fig.add_subplot(9, 1, 1);  ax2 = fig.add_subplot(9, 1, 2);  ax3 = fig.add_subplot(9, 1, 3);  ax4 = fig.add_subplot(9, 1, 4)
ax5 = fig.add_subplot(9, 1, 5);  ax6 = fig.add_subplot(9, 1, 6);  ax7 = fig.add_subplot(9, 1, 7);  ax8 = fig.add_subplot(9, 1, 8)
ax9 = fig.add_subplot(9, 1, 9)

ax1.plot(freq_1d,      EE_med,          '-y', linewidth=lw, ms=ms*4, label='EE_raw   (SID2)')
ax1.plot(freq_1d,      EE_masked,       '.b', linewidth=lw, ms=ms,   label='EE_mask  (SID2)')
ax1.plot(freq_1d_msk,  mask_list[:, 2]*1E2, '.r', linewidth=.3,  label='mask pattern')
ax1.plot(freq_1d_sid3, sid3_EE_med,     '-g', linewidth=lw*2, ms=ms,   label='EE_med   (SID3)')
ax1.plot(freq_1d_sid3, sid3_EE_ave,     '-b', linewidth=lw,   ms=ms,   label='EE_ave   (SID3)')
ax1.plot(freq_1d_sid3, sid3_EE_masked,  '-r', linewidth=lw*4, ms=ms*2, label='EE_mask  (SID3)')
#
ax2.plot(freq_1d,      EE_med,          '-y', linewidth=lw, ms=ms*4, label='EE_raw   (SID2)')
ax2.plot(freq_1d,      EE_masked,       '.b', linewidth=lw, ms=ms, label='EE_mask  (SID2)')
ax2.plot(freq_1d_msk,  mask_list[:, 2]*1E2, '.r', linewidth=.3,  label='mask pattern')
ax2.plot(freq_1d_sid3, sid3_EE_med,     '-g', linewidth=lw*2, ms=ms,   label='EE_med   (SID3)')
ax2.plot(freq_1d_sid3, sid3_EE_ave,     '-b', linewidth=lw,   ms=ms,   label='EE_ave   (SID3)')
ax2.plot(freq_1d_sid3, sid3_EE_masked,  '-r', linewidth=lw*4, ms=ms*2, label='EE_mask  (SID3)')
#
ax3.plot(freq_1d,      EE_med,          '-y', linewidth=lw, ms=ms*4, label='EE_raw   (SID2)')
ax3.plot(freq_1d,      EE_masked,       '.b', linewidth=lw, ms=ms, label='EE_mask  (SID2)')
ax3.plot(freq_1d_msk,  mask_list[:, 2]*1E2, '.r', linewidth=.3,  label='mask pattern')
ax3.plot(freq_1d_sid3, sid3_EE_med,     '-g', linewidth=lw*2, ms=ms,   label='EE_med   (SID3)')
ax3.plot(freq_1d_sid3, sid3_EE_ave,     '-b', linewidth=lw,   ms=ms,   label='EE_ave   (SID3)')
ax3.plot(freq_1d_sid3, sid3_EE_masked,  '-r', linewidth=lw*4, ms=ms*2, label='EE_mask  (SID3)')
#
ax4.plot(freq_1d,      EE_med,          '-y', linewidth=lw, ms=ms*4, label='EE_raw   (SID2)')
ax4.plot(freq_1d,      EE_masked,       '.b', linewidth=lw, ms=ms, label='EE_mask  (SID2)')
ax4.plot(freq_1d_msk,  mask_list[:, 2]*1E2, '.r', linewidth=.3,  label='mask pattern')
ax4.plot(freq_1d_sid3, sid3_EE_med,     '-g', linewidth=lw*2, ms=ms,   label='EE_med   (SID3)')
ax4.plot(freq_1d_sid3, sid3_EE_ave,     '-b', linewidth=lw,   ms=ms,   label='EE_ave   (SID3)')
ax4.plot(freq_1d_sid3, sid3_EE_masked,  '-r', linewidth=lw*4, ms=ms*2, label='EE_mask  (SID3)')
#
ax5.plot(freq_1d,      EE_med,          '-y', linewidth=lw, ms=ms*4, label='EE_raw   (SID2)')
ax5.plot(freq_1d,      EE_masked,       '.b', linewidth=lw, ms=ms, label='EE_mask  (SID2)')
ax5.plot(freq_1d_msk,  mask_list[:, 2]*1E2 ,   '.r', linewidth=.3,  label='mask pattern')
ax5.plot(freq_1d_sid3, sid3_EE_med,     '-g', linewidth=lw*2, ms=ms,   label='EE_med   (SID3)')
ax5.plot(freq_1d_sid3, sid3_EE_ave,     '-b', linewidth=lw,   ms=ms,   label='EE_ave   (SID3)')
ax5.plot(freq_1d_sid3, sid3_EE_masked,  '-r', linewidth=lw*4, ms=ms*2, label='EE_mask  (SID3)')
#
ax6.plot(freq_1d,      EE_med,          '-y', linewidth=lw, ms=ms*4, label='EE_raw   (SID2)')
ax6.plot(freq_1d,      EE_masked,       '.b', linewidth=lw, ms=ms, label='EE_mask  (SID2)')
ax6.plot(freq_1d_msk,  mask_list[:, 2]*1E2,   '.r', linewidth=.3,  label='mask pattern')
ax6.plot(freq_1d_sid3, sid3_EE_med,     '-g', linewidth=lw*2, ms=ms,   label='EE_med   (SID3)')
ax6.plot(freq_1d_sid3, sid3_EE_ave,     '-b', linewidth=lw,   ms=ms,   label='EE_ave   (SID3)')
ax6.plot(freq_1d_sid3, sid3_EE_masked,  '-r', linewidth=lw*4, ms=ms*2, label='EE_mask  (SID3)')
#
ax7.plot(freq_1d,      EE_med,          '-y', linewidth=lw, ms=ms*4, label='EE_raw   (SID2)')
ax7.plot(freq_1d,      EE_masked,       '.b', linewidth=lw, ms=ms, label='EE_mask  (SID2)')
ax7.plot(freq_1d_msk,  mask_list[:, 2]*1E2,   '.r', linewidth=.3,  label='mask pattern')
ax7.plot(freq_1d_sid3, sid3_EE_med,     '-g', linewidth=lw*2, ms=ms,   label='EE_med   (SID3)')
ax7.plot(freq_1d_sid3, sid3_EE_ave,     '-b', linewidth=lw,   ms=ms,   label='EE_ave   (SID3)')
ax7.plot(freq_1d_sid3, sid3_EE_masked,  '-r', linewidth=lw*4, ms=ms*2, label='EE_mask  (SID3)')
#
ax8.plot(freq_1d,      EE_med,          '-y', linewidth=lw, ms=ms*4, label='EE_raw   (SID2)')
ax8.plot(freq_1d,      EE_masked,       '.b', linewidth=lw, ms=ms, label='EE_mask  (SID2)')
ax8.plot(freq_1d_msk,  mask_list[:, 2]*1E2,   '.r', linewidth=.3,  label='mask pattern')
ax8.plot(freq_1d_sid3, sid3_EE_med,     '-g', linewidth=lw*2, ms=ms,   label='EE_med   (SID3)')
ax8.plot(freq_1d_sid3, sid3_EE_ave,     '-b', linewidth=lw,   ms=ms,   label='EE_ave   (SID3)')
ax8.plot(freq_1d_sid3, sid3_EE_masked,  '-r', linewidth=lw*4, ms=ms*2, label='EE_mask  (SID3)')
#
ax9.plot(freq_1d,      EE_med,          '-y', linewidth=lw, ms=ms*4, label='EE_raw   (SID2)')
ax9.plot(freq_1d,      EE_masked,       '.b', linewidth=lw, ms=ms, label='EE_mask  (SID2)')
ax9.plot(freq_1d_msk,  mask_list[:, 2]*1E2,   '.r', linewidth=.3,  label='mask pattern')
ax9.plot(freq_1d_sid3, sid3_EE_med,     '-g', linewidth=lw*2, ms=ms,   label='EE_med   (SID3)')
ax9.plot(freq_1d_sid3, sid3_EE_ave,     '-b', linewidth=lw,   ms=ms,   label='EE_ave   (SID3)')
ax9.plot(freq_1d_sid3, sid3_EE_masked,  '-r', linewidth=lw*4, ms=ms*2, label='EE_mask  (SID3)')
#
ax1.set_yscale('log');  ax2.set_yscale('log');  ax3.set_yscale('log');  ax4.set_yscale('log');  ax5.set_yscale('log')
ax6.set_yscale('log');  ax7.set_yscale('log');  ax8.set_yscale('log');  ax9.set_yscale('log')

# Label
ax1.set_ylabel(spec.str_unit); ax2.set_ylabel(spec.str_unit); ax3.set_ylabel(spec.str_unit); ax4.set_ylabel(spec.str_unit); ax5.set_ylabel(spec.str_unit)
ax6.set_ylabel(spec.str_unit); ax7.set_ylabel(spec.str_unit); ax8.set_ylabel(spec.str_unit); ax9.set_ylabel(spec.str_unit)
ax9.set_xlabel('Frequency [kHz]')
#
date1 = spec.epoch[n_sweep1];  date1 = date1.strftime(f'1st {n_sweep1}: %Y/%m/%d %R:%S   ')
date2 = spec.epoch[n_sweep2];  date2 = date2.strftime(f'Mid {n_sweep2}: %Y/%m/%d %R:%S   ')
date3 = spec.epoch[n_sweep3];  date3 = date3.strftime(f'Fin {n_sweep3}: %Y/%m/%d %R:%S')
title_date = "[ASW3.1] " + date1 + "  -  " + date2 + "  -  " + date3
ax1.set_title(title_date)

ax1.legend(loc='upper right', fontsize=8);  ax2.legend(loc='upper right', fontsize=8);  ax3.legend(loc='upper right', fontsize=8)
ax4.legend(loc='upper right', fontsize=8);  ax5.legend(loc='upper right', fontsize=8);  ax6.legend(loc='upper right', fontsize=8)
ax7.legend(loc='upper right', fontsize=8);  ax8.legend(loc='upper right', fontsize=8);  ax9.legend(loc='upper right', fontsize=8)

# range: X-axis
xlim=[f_min,       f_max];        ax1.set_xlim(xlim);  xlim=[f_min+5000,  f_max+5000];   ax2.set_xlim(xlim);  
xlim=[f_min+10000, f_max+10000];  ax3.set_xlim(xlim);  xlim=[f_min+15000, f_max+15000];  ax4.set_xlim(xlim)
xlim=[f_min+20000, f_max+20000];  ax5.set_xlim(xlim);  xlim=[f_min+25000, f_max+25000];  ax6.set_xlim(xlim)
xlim=[f_min+30000, f_max+30000];  ax7.set_xlim(xlim);  xlim=[f_min+35000, f_max+35000];  ax8.set_xlim(xlim)
xlim=[f_min+40000, f_max+40000];  ax9.set_xlim(xlim)
ax1.tick_params(labelsize=7); ax2.tick_params(labelsize=7); ax3.tick_params(labelsize=7); ax4.tick_params(labelsize=7); ax5.tick_params(labelsize=7)
ax6.tick_params(labelsize=7); ax7.tick_params(labelsize=7); ax8.tick_params(labelsize=7); ax9.tick_params(labelsize=7)

ylim=[10**p_min, 10**p_max]
ax1.set_ylim(ylim); ax2.set_ylim(ylim);  ax3.set_ylim(ylim); ax4.set_ylim(ylim); ax5.set_ylim(ylim); ax6.set_ylim(ylim);  
ax7.set_ylim(ylim); ax8.set_ylim(ylim);  ax9.set_ylim(ylim)

fig.subplots_adjust(hspace=0)
fig.show
png_fname = work_dir+data_name+'_N_ASW3.1.png';  fig.savefig(png_fname)


# SID-3 Frequency table - ASW3.2

In [ ]:
asw32_ver = 3.2
freq_1d_sid3, f_step_sid3, f_width_sid3 = juice_cdf._frequency_sid3(asw32_ver);  n_freq_sid3 = len(freq_1d_sid3)

print( f"  sid: {asw32_ver}\tn_freq_mask: {n_freq_msk}")
print( f"SID-2: {freq_1d.shape[0]:5d}\tfreq: {freq_1d[0]:.5f} ({freq_1d[0]-freq_w_1d[0]/2:.5f} - {freq_1d[0]+freq_w_1d[0]/2:.5f}) - {freq_1d[-1]:.5f} ({freq_1d[-1]-freq_w_1d[-1]/2:.5f} - {freq_1d[-1]+freq_w_1d[-1]/2:.5f}) [kHz]  \tstep: {freq_w_1d[0]:.5f} - {freq_w_1d[-1]:.5f}")
print( f"SID-3: {freq_1d_sid3.shape[0]:5d}\tfreq: {freq_1d_sid3[0]:.5f} ({freq_1d_sid3[0]-f_step_sid3[0]/2:.5f} - {freq_1d_sid3[0]+f_step_sid3[0]/2:.5f}) - {freq_1d_sid3[-1]:.5f} ({freq_1d_sid3[-1]-f_step_sid3[-1]/2:.5f} - {freq_1d_sid3[-1]+f_step_sid3[-1]/2:.5f}) [kHz]  \tstep: {f_step_sid3[0]:.5f} - {f_step_sid3[-1]:.5f}")

In [ ]:
# TABLE to SID-3 (n_freq_sid3, 3):    0 - start MASK-ch, 1 - end MASK-ch,  2 - width MASK-ch
tbl_msk_to_sid3  = juice_cdf._frequency_sid2_to_data(freq_1d_sid3, f_step_sid3, freq_1d_msk, freq_w_msk)
for i in range(n_freq_sid3):
    i0 = tbl_msk_to_sid3[i][0];  i1 = tbl_msk_to_sid3[i][1];  i2 = tbl_msk_to_sid3[i][2]
    num = 0
    for j in range (i0, i1+1):
        if mask_list[j, 2] == 0:  num += 1.0

In [ ]:
# masked SID-2
EE_masked = copy.deepcopy(EE_med)
j0        = 0
for i in range(n_freq_msk):
    if mask_list[i, 2] > 0:
        for j in range(j0, n_freq1):
            if freq_1d[j] >  mask_list[i, 1]:
                break
            if freq_1d[j] >= mask_list[i, 0] and freq_1d[j] < mask_list[i, 1]:
                EE_masked[j] = math.nan
        j0 = j

# masked SID-3
sid3_EE_med = np.zeros(n_freq_sid3); sid3_EE_ave = np.zeros(n_freq_sid3); sid3_EE_masked = np.zeros(n_freq_sid3)     # median / masked
tbl_sid2_to_sid3  = juice_cdf._frequency_sid2_to_data(freq_1d_sid3, f_step_sid3, freq_1d, freq_w_1d)
for j in range(n_freq_sid3):
    tbl = tbl_sid2_to_sid3[j]                        # 0 - start ch, 1 - end ch, 2 - width ch
    sid3_EE_med   [j] = np.nanmedian( EE_med   [tbl[0]:tbl[1]+1] )
    sid3_EE_ave   [j] = np.nanmean  ( EE_med   [tbl[0]:tbl[1]+1] )
    sid3_EE_masked[j] = np.nanmean  ( EE_masked[tbl[0]:tbl[1]+1] )
    if np.sum(np.isnan(EE_masked[tbl[0]:tbl[1]+1]))/tbl[2] >= 0.5:
        print(j, freq_1d_sid3[j], tbl[2], np.sum(np.isnan(EE_masked[tbl[0]:tbl[1]+1])), np.sum(np.isnan(EE_masked[tbl[0]:tbl[1]+1]))/tbl[2]*100)

In [ ]:
p_min = p_min2;  p_max = p_max2
f_min = f_min1;  f_max = f_max1
if f_mode == 0:
    f_min = f_mode_min;  f_max = f_mode_max
lw = 1;  ms = 3

fig = plt.figure(figsize=(14, 11));  ax1 = fig.add_subplot(1, 1, 1)
ax1.plot(freq_1d,       EE_med,          '.y', linewidth=lw,   ms=ms,   label='EE_raw   (SID2)')
ax1.plot(freq_1d,       EE_masked,       '.k', linewidth=lw,   ms=ms,   label='EE_mask  (SID2)')
ax1.plot(freq_1d_sid3,  sid3_EE_med,     '-g', linewidth=lw*2, ms=ms,   label='EE_med   (SID3)')
ax1.plot(freq_1d_sid3,  sid3_EE_ave,     '-b', linewidth=lw,   ms=ms,   label='EE_ave   (SID3)')
ax1.plot(freq_1d_sid3,  sid3_EE_masked,  '-r', linewidth=lw*4, ms=ms*2, label='EE_mask  (SID3)')
ax1.set_yscale('log')
if f_mode == 1:                   ax1.set_xscale('log')
xlim=[f_min, f_max];              ax1.set_xlim(xlim)
ylim=[10**(p_min), 10**(p_max)];  ax1.set_ylim(ylim)

ax1.set_ylabel('EE '+spec.str_unit)
date1 = spec.epoch[n_sweep1];  date1 = date1.strftime('1st: %Y/%m/%d %R:%S   ')
date2 = spec.epoch[n_sweep2];  date2 = date2.strftime('Mid: %Y/%m/%d %R:%S   ')
date3 = spec.epoch[n_sweep3];  date3 = date3.strftime('Fin: %Y/%m/%d %R:%S')
title_date = "[ASW3.2] " + date1 + "  -  " + date2 + "  -  " + date3
ax1.set_title(title_date);   ax1.legend(loc='upper right', fontsize=8)
plt.subplots_adjust(hspace=.03)
fig.show
png_fname = work_dir+data_name+'_W_ASW3.2.png';  fig.savefig(png_fname)

In [ ]:
p_min = p_min2;   p_max = p_max2
f_min = 0;        f_max = 5000

fig = plt.figure(figsize=(14, 20))
ax1 = fig.add_subplot(9, 1, 1);  ax2 = fig.add_subplot(9, 1, 2);  ax3 = fig.add_subplot(9, 1, 3);  ax4 = fig.add_subplot(9, 1, 4)
ax5 = fig.add_subplot(9, 1, 5);  ax6 = fig.add_subplot(9, 1, 6);  ax7 = fig.add_subplot(9, 1, 7);  ax8 = fig.add_subplot(9, 1, 8)
ax9 = fig.add_subplot(9, 1, 9)

ax1.plot(freq_1d,      EE_med,          '-y', linewidth=lw, ms=ms*4, label='EE_raw   (SID2)')
ax1.plot(freq_1d,      EE_masked,       '.b', linewidth=lw, ms=ms,   label='EE_mask  (SID2)')
ax1.plot(freq_1d_msk,  mask_list[:, 2]*1E2, '.r', linewidth=.3,  label='mask pattern')
ax1.plot(freq_1d_sid3, sid3_EE_med,     '-g', linewidth=lw*2, ms=ms,   label='EE_med   (SID3)')
ax1.plot(freq_1d_sid3, sid3_EE_ave,     '-b', linewidth=lw,   ms=ms,   label='EE_ave   (SID3)')
ax1.plot(freq_1d_sid3, sid3_EE_masked,  '-r', linewidth=lw*4, ms=ms*2, label='EE_mask  (SID3)')
#
ax2.plot(freq_1d,      EE_med,          '-y', linewidth=lw, ms=ms*4, label='EE_raw   (SID2)')
ax2.plot(freq_1d,      EE_masked,       '.b', linewidth=lw, ms=ms, label='EE_mask  (SID2)')
ax2.plot(freq_1d_msk,  mask_list[:, 2]*1E2, '.r', linewidth=.3,  label='mask pattern')
ax2.plot(freq_1d_sid3, sid3_EE_med,     '-g', linewidth=lw*2, ms=ms,   label='EE_med   (SID3)')
ax2.plot(freq_1d_sid3, sid3_EE_ave,     '-b', linewidth=lw,   ms=ms,   label='EE_ave   (SID3)')
ax2.plot(freq_1d_sid3, sid3_EE_masked,  '-r', linewidth=lw*4, ms=ms*2, label='EE_mask  (SID3)')
#
ax3.plot(freq_1d,      EE_med,          '-y', linewidth=lw, ms=ms*4, label='EE_raw   (SID2)')
ax3.plot(freq_1d,      EE_masked,       '.b', linewidth=lw, ms=ms, label='EE_mask  (SID2)')
ax3.plot(freq_1d_msk,  mask_list[:, 2]*1E2, '.r', linewidth=.3,  label='mask pattern')
ax3.plot(freq_1d_sid3, sid3_EE_med,     '-g', linewidth=lw*2, ms=ms,   label='EE_med   (SID3)')
ax3.plot(freq_1d_sid3, sid3_EE_ave,     '-b', linewidth=lw,   ms=ms,   label='EE_ave   (SID3)')
ax3.plot(freq_1d_sid3, sid3_EE_masked,  '-r', linewidth=lw*4, ms=ms*2, label='EE_mask  (SID3)')
#
ax4.plot(freq_1d,      EE_med,          '-y', linewidth=lw, ms=ms*4, label='EE_raw   (SID2)')
ax4.plot(freq_1d,      EE_masked,       '.b', linewidth=lw, ms=ms, label='EE_mask  (SID2)')
ax4.plot(freq_1d_msk,  mask_list[:, 2]*1E2, '.r', linewidth=.3,  label='mask pattern')
ax4.plot(freq_1d_sid3, sid3_EE_med,     '-g', linewidth=lw*2, ms=ms,   label='EE_med   (SID3)')
ax4.plot(freq_1d_sid3, sid3_EE_ave,     '-b', linewidth=lw,   ms=ms,   label='EE_ave   (SID3)')
ax4.plot(freq_1d_sid3, sid3_EE_masked,  '-r', linewidth=lw*4, ms=ms*2, label='EE_mask  (SID3)')
#
ax5.plot(freq_1d,      EE_med,          '-y', linewidth=lw, ms=ms*4, label='EE_raw   (SID2)')
ax5.plot(freq_1d,      EE_masked,       '.b', linewidth=lw, ms=ms, label='EE_mask  (SID2)')
ax5.plot(freq_1d_msk,  mask_list[:, 2]*1E2 ,   '.r', linewidth=.3,  label='mask pattern')
ax5.plot(freq_1d_sid3, sid3_EE_med,     '-g', linewidth=lw*2, ms=ms,   label='EE_med   (SID3)')
ax5.plot(freq_1d_sid3, sid3_EE_ave,     '-b', linewidth=lw,   ms=ms,   label='EE_ave   (SID3)')
ax5.plot(freq_1d_sid3, sid3_EE_masked,  '-r', linewidth=lw*4, ms=ms*2, label='EE_mask  (SID3)')
#
ax6.plot(freq_1d,      EE_med,          '-y', linewidth=lw, ms=ms*4, label='EE_raw   (SID2)')
ax6.plot(freq_1d,      EE_masked,       '.b', linewidth=lw, ms=ms, label='EE_mask  (SID2)')
ax6.plot(freq_1d_msk,  mask_list[:, 2]*1E2,   '.r', linewidth=.3,  label='mask pattern')
ax6.plot(freq_1d_sid3, sid3_EE_med,     '-g', linewidth=lw*2, ms=ms,   label='EE_med   (SID3)')
ax6.plot(freq_1d_sid3, sid3_EE_ave,     '-b', linewidth=lw,   ms=ms,   label='EE_ave   (SID3)')
ax6.plot(freq_1d_sid3, sid3_EE_masked,  '-r', linewidth=lw*4, ms=ms*2, label='EE_mask  (SID3)')
#
ax7.plot(freq_1d,      EE_med,          '-y', linewidth=lw, ms=ms*4, label='EE_raw   (SID2)')
ax7.plot(freq_1d,      EE_masked,       '.b', linewidth=lw, ms=ms, label='EE_mask  (SID2)')
ax7.plot(freq_1d_msk,  mask_list[:, 2]*1E2,   '.r', linewidth=.3,  label='mask pattern')
ax7.plot(freq_1d_sid3, sid3_EE_med,     '-g', linewidth=lw*2, ms=ms,   label='EE_med   (SID3)')
ax7.plot(freq_1d_sid3, sid3_EE_ave,     '-b', linewidth=lw,   ms=ms,   label='EE_ave   (SID3)')
ax7.plot(freq_1d_sid3, sid3_EE_masked,  '-r', linewidth=lw*4, ms=ms*2, label='EE_mask  (SID3)')
#
ax8.plot(freq_1d,      EE_med,          '-y', linewidth=lw, ms=ms*4, label='EE_raw   (SID2)')
ax8.plot(freq_1d,      EE_masked,       '.b', linewidth=lw, ms=ms, label='EE_mask  (SID2)')
ax8.plot(freq_1d_msk,  mask_list[:, 2]*1E2,   '.r', linewidth=.3,  label='mask pattern')
ax8.plot(freq_1d_sid3, sid3_EE_med,     '-g', linewidth=lw*2, ms=ms,   label='EE_med   (SID3)')
ax8.plot(freq_1d_sid3, sid3_EE_ave,     '-b', linewidth=lw,   ms=ms,   label='EE_ave   (SID3)')
ax8.plot(freq_1d_sid3, sid3_EE_masked,  '-r', linewidth=lw*4, ms=ms*2, label='EE_mask  (SID3)')
#
ax9.plot(freq_1d,      EE_med,          '-y', linewidth=lw, ms=ms*4, label='EE_raw   (SID2)')
ax9.plot(freq_1d,      EE_masked,       '.b', linewidth=lw, ms=ms, label='EE_mask  (SID2)')
ax9.plot(freq_1d_msk,  mask_list[:, 2]*1E2,   '.r', linewidth=.3,  label='mask pattern')
ax9.plot(freq_1d_sid3, sid3_EE_med,     '-g', linewidth=lw*2, ms=ms,   label='EE_med   (SID3)')
ax9.plot(freq_1d_sid3, sid3_EE_ave,     '-b', linewidth=lw,   ms=ms,   label='EE_ave   (SID3)')
ax9.plot(freq_1d_sid3, sid3_EE_masked,  '-r', linewidth=lw*4, ms=ms*2, label='EE_mask  (SID3)')
#
ax1.set_yscale('log');  ax2.set_yscale('log');  ax3.set_yscale('log');  ax4.set_yscale('log');  ax5.set_yscale('log')
ax6.set_yscale('log');  ax7.set_yscale('log');  ax8.set_yscale('log');  ax9.set_yscale('log')

# Label
ax1.set_ylabel(spec.str_unit); ax2.set_ylabel(spec.str_unit); ax3.set_ylabel(spec.str_unit); ax4.set_ylabel(spec.str_unit); ax5.set_ylabel(spec.str_unit)
ax6.set_ylabel(spec.str_unit); ax7.set_ylabel(spec.str_unit); ax8.set_ylabel(spec.str_unit); ax9.set_ylabel(spec.str_unit)
ax9.set_xlabel('Frequency [kHz]')
#
date1 = spec.epoch[n_sweep1];  date1 = date1.strftime(f'1st {n_sweep1}: %Y/%m/%d %R:%S   ')
date2 = spec.epoch[n_sweep2];  date2 = date2.strftime(f'Mid {n_sweep2}: %Y/%m/%d %R:%S   ')
date3 = spec.epoch[n_sweep3];  date3 = date3.strftime(f'Fin {n_sweep3}: %Y/%m/%d %R:%S')
title_date = "[ASW3.2] " + date1 + "  -  " + date2 + "  -  " + date3
ax1.set_title(title_date)

ax1.legend(loc='upper right', fontsize=8);  ax2.legend(loc='upper right', fontsize=8);  ax3.legend(loc='upper right', fontsize=8)
ax4.legend(loc='upper right', fontsize=8);  ax5.legend(loc='upper right', fontsize=8);  ax6.legend(loc='upper right', fontsize=8)
ax7.legend(loc='upper right', fontsize=8);  ax8.legend(loc='upper right', fontsize=8);  ax9.legend(loc='upper right', fontsize=8)

# range: X-axis
xlim=[f_min,       f_max];        ax1.set_xlim(xlim);  xlim=[f_min+5000,  f_max+5000];   ax2.set_xlim(xlim);  
xlim=[f_min+10000, f_max+10000];  ax3.set_xlim(xlim);  xlim=[f_min+15000, f_max+15000];  ax4.set_xlim(xlim)
xlim=[f_min+20000, f_max+20000];  ax5.set_xlim(xlim);  xlim=[f_min+25000, f_max+25000];  ax6.set_xlim(xlim)
xlim=[f_min+30000, f_max+30000];  ax7.set_xlim(xlim);  xlim=[f_min+35000, f_max+35000];  ax8.set_xlim(xlim)
xlim=[f_min+40000, f_max+40000];  ax9.set_xlim(xlim)
ax1.tick_params(labelsize=7); ax2.tick_params(labelsize=7); ax3.tick_params(labelsize=7); ax4.tick_params(labelsize=7); ax5.tick_params(labelsize=7)
ax6.tick_params(labelsize=7); ax7.tick_params(labelsize=7); ax8.tick_params(labelsize=7); ax9.tick_params(labelsize=7)

ylim=[10**p_min, 10**p_max]
ax1.set_ylim(ylim); ax2.set_ylim(ylim);  ax3.set_ylim(ylim); ax4.set_ylim(ylim); ax5.set_ylim(ylim); ax6.set_ylim(ylim);  
ax7.set_ylim(ylim); ax8.set_ylim(ylim);  ax9.set_ylim(ylim)

fig.subplots_adjust(hspace=0)
fig.show
png_fname = work_dir+data_name+'_N_ASW3.2.png';  fig.savefig(png_fname)

# TEST: ASW3.1

In [ ]:
asw31_ver = 3.2
freq_1d_sid3, f_step_sid3, f_width_sid3 = juice_cdf._frequency_sid3(asw31_ver);  n_freq_sid3 = len(freq_1d_sid3)

print( f"  sid: {asw31_ver}\tn_freq_mask: {n_freq_msk}")
print( f"SID-2: {freq_1d.shape[0]:5d}\tfreq: {freq_1d[0]:.5f} ({freq_1d[0]-freq_w_1d[0]/2:.5f} - {freq_1d[0]+freq_w_1d[0]/2:.5f}) - {freq_1d[-1]:.5f} ({freq_1d[-1]-freq_w_1d[-1]/2:.5f} - {freq_1d[-1]+freq_w_1d[-1]/2:.5f}) [kHz]  \tstep: {freq_w_1d[0]:.5f} - {freq_w_1d[-1]:.5f}")
print( f"SID-3: {freq_1d_sid3.shape[0]:5d}\tfreq: {freq_1d_sid3[0]:.5f} ({freq_1d_sid3[0]-f_step_sid3[0]/2:.5f} - {freq_1d_sid3[0]+f_step_sid3[0]/2:.5f}) - {freq_1d_sid3[-1]:.5f} ({freq_1d_sid3[-1]-f_step_sid3[-1]/2:.5f} - {freq_1d_sid3[-1]+f_step_sid3[-1]/2:.5f}) [kHz]  \tstep: {f_step_sid3[0]:.5f} - {f_step_sid3[-1]:.5f}")

In [ ]:
# TABLE to SID-3 (n_freq_sid3, 3):    0 - start MASK-ch, 1 - end MASK-ch,  2 - width MASK-ch
tbl_msk_to_sid3  = juice_cdf._frequency_sid2_to_data(freq_1d_sid3, f_step_sid3, freq_1d_msk, freq_w_msk)
for i in range(n_freq_sid3):
    i0 = tbl_msk_to_sid3[i][0];  i1 = tbl_msk_to_sid3[i][1];  i2 = tbl_msk_to_sid3[i][2]
    num = 0
    for j in range (i0, i1+1):
        if mask_list[j, 2] == 0:  num += 1.0

In [ ]:
# masked SID-2
EE_masked = copy.deepcopy(EE_med)
j0        = 0
for i in range(n_freq_msk):
    if mask_list[i, 2] > 0:
        for j in range(j0, n_freq1):
            if freq_1d[j] >  mask_list[i, 1]:
                break
            if freq_1d[j] >= mask_list[i, 0] and freq_1d[j] < mask_list[i, 1]:
                EE_masked[j] = math.nan
        j0 = j

# masked SID-3
sid3_EE_med = np.zeros(n_freq_sid3); sid3_EE_ave = np.zeros(n_freq_sid3); sid3_EE_masked = np.zeros(n_freq_sid3)     # median / masked
tbl_sid2_to_sid3  = juice_cdf._frequency_sid2_to_data(freq_1d_sid3, f_step_sid3, freq_1d, freq_w_1d)
for j in range(n_freq_sid3):
    tbl = tbl_sid2_to_sid3[j]                        # 0 - start ch, 1 - end ch, 2 - width ch
    sid3_EE_med   [j] = np.nanmedian( EE_med   [tbl[0]:tbl[1]+1] )
    sid3_EE_ave   [j] = np.nanmean  ( EE_med   [tbl[0]:tbl[1]+1] )
    sid3_EE_masked[j] = np.nanmean  ( EE_masked[tbl[0]:tbl[1]+1] )
    if np.sum(np.isnan(EE_masked[tbl[0]:tbl[1]+1]))/tbl[2] >= 0.5:
        print(j, freq_1d_sid3[j], tbl[2], np.sum(np.isnan(EE_masked[tbl[0]:tbl[1]+1])), np.sum(np.isnan(EE_masked[tbl[0]:tbl[1]+1]))/tbl[2]*100)

In [ ]:
p_min = p_min2;  p_max = p_max2
f_min = f_min1;  f_max = f_max1
# f_min = 40000;   # f_max = 50
if f_mode == 0:
    f_min = f_mode_min;  f_max = f_mode_max
lw = 1;  ms = 3

fig = plt.figure(figsize=(14, 11));  ax1 = fig.add_subplot(1, 1, 1)
ax1.plot(freq_1d_sid3,  sid3_EE_med,     '-g', linewidth=lw*2, ms=ms,   label='EE_med   (SID3)')
ax1.plot(freq_1d_sid3,  sid3_EE_ave,     '-b', linewidth=lw,   ms=ms,   label='EE_ave   (SID3)')
ax1.plot(freq_1d_sid3,  sid3_EE_masked,  '-r', linewidth=lw*4, ms=ms*2, label='EE_mask  (SID3)')
ax1.plot(freq_1d,       EE_med,          '.y', linewidth=lw,   ms=ms,   label='EE_raw   (SID2)')
ax1.plot(freq_1d,       EE_masked,       '.k', linewidth=lw,   ms=ms,   label='EE_mask  (SID2)')
ax1.set_yscale('log')
if f_mode == 1:                   ax1.set_xscale('log')
xlim=[f_min, f_max];              ax1.set_xlim(xlim)
ylim=[10**(p_min), 10**(p_max)];  ax1.set_ylim(ylim)

ax1.set_ylabel('EE '+spec.str_unit)
date1 = spec.epoch[n_sweep1];  date1 = date1.strftime('1st: %Y/%m/%d %R:%S   ')
date2 = spec.epoch[n_sweep2];  date2 = date2.strftime('Mid: %Y/%m/%d %R:%S   ')
date3 = spec.epoch[n_sweep3];  date3 = date3.strftime('Fin: %Y/%m/%d %R:%S')
title_date = "[ASW3.1] " + date1 + "  -  " + date2 + "  -  " + date3
ax1.set_title(title_date);   ax1.legend(loc='upper right', fontsize=8)
plt.subplots_adjust(hspace=.03)
fig.show
png_fname = work_dir+data_name+'_W_ASWtest.png';  fig.savefig(png_fname)

In [ ]:
p_min = p_min2;   p_max = p_max2
f_min = 0;        f_max = 5000

fig = plt.figure(figsize=(14, 20))
ax1 = fig.add_subplot(9, 1, 1);  ax2 = fig.add_subplot(9, 1, 2);  ax3 = fig.add_subplot(9, 1, 3);  ax4 = fig.add_subplot(9, 1, 4)
ax5 = fig.add_subplot(9, 1, 5);  ax6 = fig.add_subplot(9, 1, 6);  ax7 = fig.add_subplot(9, 1, 7);  ax8 = fig.add_subplot(9, 1, 8)
ax9 = fig.add_subplot(9, 1, 9)

ax1.plot(freq_1d,      EE_med,          '-y', linewidth=lw, ms=ms*4, label='EE_raw   (SID2)')
ax1.plot(freq_1d,      EE_masked,       '.b', linewidth=lw, ms=ms,   label='EE_mask  (SID2)')
ax1.plot(freq_1d_msk,  mask_list[:, 2]*1E2, '.r', linewidth=.3,  label='mask pattern')
ax1.plot(freq_1d_sid3, sid3_EE_med,     '-g', linewidth=lw*2, ms=ms,   label='EE_med   (SID3)')
ax1.plot(freq_1d_sid3, sid3_EE_ave,     '-b', linewidth=lw,   ms=ms,   label='EE_ave   (SID3)')
ax1.plot(freq_1d_sid3, sid3_EE_masked,  '-r', linewidth=lw*4, ms=ms*2, label='EE_mask  (SID3)')
#
ax2.plot(freq_1d,      EE_med,          '-y', linewidth=lw, ms=ms*4, label='EE_raw   (SID2)')
ax2.plot(freq_1d,      EE_masked,       '.b', linewidth=lw, ms=ms, label='EE_mask  (SID2)')
ax2.plot(freq_1d_msk,  mask_list[:, 2]*1E2, '.r', linewidth=.3,  label='mask pattern')
ax2.plot(freq_1d_sid3, sid3_EE_med,     '-g', linewidth=lw*2, ms=ms,   label='EE_med   (SID3)')
ax2.plot(freq_1d_sid3, sid3_EE_ave,     '-b', linewidth=lw,   ms=ms,   label='EE_ave   (SID3)')
ax2.plot(freq_1d_sid3, sid3_EE_masked,  '-r', linewidth=lw*4, ms=ms*2, label='EE_mask  (SID3)')
#
ax3.plot(freq_1d,      EE_med,          '-y', linewidth=lw, ms=ms*4, label='EE_raw   (SID2)')
ax3.plot(freq_1d,      EE_masked,       '.b', linewidth=lw, ms=ms, label='EE_mask  (SID2)')
ax3.plot(freq_1d_msk,  mask_list[:, 2]*1E2, '.r', linewidth=.3,  label='mask pattern')
ax3.plot(freq_1d_sid3, sid3_EE_med,     '-g', linewidth=lw*2, ms=ms,   label='EE_med   (SID3)')
ax3.plot(freq_1d_sid3, sid3_EE_ave,     '-b', linewidth=lw,   ms=ms,   label='EE_ave   (SID3)')
ax3.plot(freq_1d_sid3, sid3_EE_masked,  '-r', linewidth=lw*4, ms=ms*2, label='EE_mask  (SID3)')
#
ax4.plot(freq_1d,      EE_med,          '-y', linewidth=lw, ms=ms*4, label='EE_raw   (SID2)')
ax4.plot(freq_1d,      EE_masked,       '.b', linewidth=lw, ms=ms, label='EE_mask  (SID2)')
ax4.plot(freq_1d_msk,  mask_list[:, 2]*1E2, '.r', linewidth=.3,  label='mask pattern')
ax4.plot(freq_1d_sid3, sid3_EE_med,     '-g', linewidth=lw*2, ms=ms,   label='EE_med   (SID3)')
ax4.plot(freq_1d_sid3, sid3_EE_ave,     '-b', linewidth=lw,   ms=ms,   label='EE_ave   (SID3)')
ax4.plot(freq_1d_sid3, sid3_EE_masked,  '-r', linewidth=lw*4, ms=ms*2, label='EE_mask  (SID3)')
#
ax5.plot(freq_1d,      EE_med,          '-y', linewidth=lw, ms=ms*4, label='EE_raw   (SID2)')
ax5.plot(freq_1d,      EE_masked,       '.b', linewidth=lw, ms=ms, label='EE_mask  (SID2)')
ax5.plot(freq_1d_msk,  mask_list[:, 2]*1E2 ,   '.r', linewidth=.3,  label='mask pattern')
ax5.plot(freq_1d_sid3, sid3_EE_med,     '-g', linewidth=lw*2, ms=ms,   label='EE_med   (SID3)')
ax5.plot(freq_1d_sid3, sid3_EE_ave,     '-b', linewidth=lw,   ms=ms,   label='EE_ave   (SID3)')
ax5.plot(freq_1d_sid3, sid3_EE_masked,  '-r', linewidth=lw*4, ms=ms*2, label='EE_mask  (SID3)')
#
ax6.plot(freq_1d,      EE_med,          '-y', linewidth=lw, ms=ms*4, label='EE_raw   (SID2)')
ax6.plot(freq_1d,      EE_masked,       '.b', linewidth=lw, ms=ms, label='EE_mask  (SID2)')
ax6.plot(freq_1d_msk,  mask_list[:, 2]*1E2,   '.r', linewidth=.3,  label='mask pattern')
ax6.plot(freq_1d_sid3, sid3_EE_med,     '-g', linewidth=lw*2, ms=ms,   label='EE_med   (SID3)')
ax6.plot(freq_1d_sid3, sid3_EE_ave,     '-b', linewidth=lw,   ms=ms,   label='EE_ave   (SID3)')
ax6.plot(freq_1d_sid3, sid3_EE_masked,  '-r', linewidth=lw*4, ms=ms*2, label='EE_mask  (SID3)')
#
ax7.plot(freq_1d,      EE_med,          '-y', linewidth=lw, ms=ms*4, label='EE_raw   (SID2)')
ax7.plot(freq_1d,      EE_masked,       '.b', linewidth=lw, ms=ms, label='EE_mask  (SID2)')
ax7.plot(freq_1d_msk,  mask_list[:, 2]*1E2,   '.r', linewidth=.3,  label='mask pattern')
ax7.plot(freq_1d_sid3, sid3_EE_med,     '-g', linewidth=lw*2, ms=ms,   label='EE_med   (SID3)')
ax7.plot(freq_1d_sid3, sid3_EE_ave,     '-b', linewidth=lw,   ms=ms,   label='EE_ave   (SID3)')
ax7.plot(freq_1d_sid3, sid3_EE_masked,  '-r', linewidth=lw*4, ms=ms*2, label='EE_mask  (SID3)')
#
ax8.plot(freq_1d,      EE_med,          '-y', linewidth=lw, ms=ms*4, label='EE_raw   (SID2)')
ax8.plot(freq_1d,      EE_masked,       '.b', linewidth=lw, ms=ms, label='EE_mask  (SID2)')
ax8.plot(freq_1d_msk,  mask_list[:, 2]*1E2,   '.r', linewidth=.3,  label='mask pattern')
ax8.plot(freq_1d_sid3, sid3_EE_med,     '-g', linewidth=lw*2, ms=ms,   label='EE_med   (SID3)')
ax8.plot(freq_1d_sid3, sid3_EE_ave,     '-b', linewidth=lw,   ms=ms,   label='EE_ave   (SID3)')
ax8.plot(freq_1d_sid3, sid3_EE_masked,  '-r', linewidth=lw*4, ms=ms*2, label='EE_mask  (SID3)')
#
ax9.plot(freq_1d,      EE_med,          '-y', linewidth=lw, ms=ms*4, label='EE_raw   (SID2)')
ax9.plot(freq_1d,      EE_masked,       '.b', linewidth=lw, ms=ms, label='EE_mask  (SID2)')
ax9.plot(freq_1d_msk,  mask_list[:, 2]*1E2,   '.r', linewidth=.3,  label='mask pattern')
ax9.plot(freq_1d_sid3, sid3_EE_med,     '-g', linewidth=lw*2, ms=ms,   label='EE_med   (SID3)')
ax9.plot(freq_1d_sid3, sid3_EE_ave,     '-b', linewidth=lw,   ms=ms,   label='EE_ave   (SID3)')
ax9.plot(freq_1d_sid3, sid3_EE_masked,  '-r', linewidth=lw*4, ms=ms*2, label='EE_mask  (SID3)')
#
ax1.set_yscale('log');  ax2.set_yscale('log');  ax3.set_yscale('log');  ax4.set_yscale('log');  ax5.set_yscale('log')
ax6.set_yscale('log');  ax7.set_yscale('log');  ax8.set_yscale('log');  ax9.set_yscale('log')

# Label
ax1.set_ylabel(spec.str_unit); ax2.set_ylabel(spec.str_unit); ax3.set_ylabel(spec.str_unit); ax4.set_ylabel(spec.str_unit); ax5.set_ylabel(spec.str_unit)
ax6.set_ylabel(spec.str_unit); ax7.set_ylabel(spec.str_unit); ax8.set_ylabel(spec.str_unit); ax9.set_ylabel(spec.str_unit)
ax9.set_xlabel('Frequency [kHz]')
#
date1 = spec.epoch[n_sweep1];  date1 = date1.strftime(f'1st {n_sweep1}: %Y/%m/%d %R:%S   ')
date2 = spec.epoch[n_sweep2];  date2 = date2.strftime(f'Mid {n_sweep2}: %Y/%m/%d %R:%S   ')
date3 = spec.epoch[n_sweep3];  date3 = date3.strftime(f'Fin {n_sweep3}: %Y/%m/%d %R:%S')
title_date = "[ASW3.1] " + date1 + "  -  " + date2 + "  -  " + date3
ax1.set_title(title_date)

ax1.legend(loc='upper right', fontsize=8);  ax2.legend(loc='upper right', fontsize=8);  ax3.legend(loc='upper right', fontsize=8)
ax4.legend(loc='upper right', fontsize=8);  ax5.legend(loc='upper right', fontsize=8);  ax6.legend(loc='upper right', fontsize=8)
ax7.legend(loc='upper right', fontsize=8);  ax8.legend(loc='upper right', fontsize=8);  ax9.legend(loc='upper right', fontsize=8)

# range: X-axis
xlim=[f_min,       f_max];        ax1.set_xlim(xlim);  xlim=[f_min+5000,  f_max+5000];   ax2.set_xlim(xlim);  
xlim=[f_min+10000, f_max+10000];  ax3.set_xlim(xlim);  xlim=[f_min+15000, f_max+15000];  ax4.set_xlim(xlim)
xlim=[f_min+20000, f_max+20000];  ax5.set_xlim(xlim);  xlim=[f_min+25000, f_max+25000];  ax6.set_xlim(xlim)
xlim=[f_min+30000, f_max+30000];  ax7.set_xlim(xlim);  xlim=[f_min+35000, f_max+35000];  ax8.set_xlim(xlim)
xlim=[f_min+40000, f_max+40000];  ax9.set_xlim(xlim)
ax1.tick_params(labelsize=7); ax2.tick_params(labelsize=7); ax3.tick_params(labelsize=7); ax4.tick_params(labelsize=7); ax5.tick_params(labelsize=7)
ax6.tick_params(labelsize=7); ax7.tick_params(labelsize=7); ax8.tick_params(labelsize=7); ax9.tick_params(labelsize=7)

ylim=[10**p_min, 10**p_max]
ax1.set_ylim(ylim); ax2.set_ylim(ylim);  ax3.set_ylim(ylim); ax4.set_ylim(ylim); ax5.set_ylim(ylim); ax6.set_ylim(ylim);  
ax7.set_ylim(ylim); ax8.set_ylim(ylim);  ax9.set_ylim(ylim)

fig.subplots_adjust(hspace=0)
fig.show
png_fname = work_dir+data_name+'_N_ASW3test.png';  fig.savefig(png_fname)
